# AutoQSAR Benchmark Summary

This notebook reads the output from `run_autoqsar_ga_benchmarks.py` and summarizes model performance across datasets.

By default it loads the most recent run under `./benchmark_results/`, then builds:
- a model-performance summary across datasets
- a leaderboard-relative summary using cached top-10 reference artifacts from the run directory
- diagnostic visuals for model spread, predictions, and (if present) GA convergence
- feature-family selection representation analysis from selector outputs (coverage, enrichment, and drop/watchlist suggestions)



In [66]:
# @title 0. Choose a benchmark run { display-mode: "form" }
benchmark_run_dir = "AUTO"  # @param {type:"string"}
show_top_n_models = 10  # @param {type:"slider", min:3, max:20, step:1}
rmse_metric_column_preference = "AUTO"  # @param {type:"string"}
prediction_sample_cap = 5000  # @param {type:"integer"}


In [67]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


def display_note(text):
    display(Markdown(text))


def resolve_benchmark_dir(path_text="AUTO"):
    path_text = str(path_text).strip()
    if path_text and path_text.upper() != "AUTO":
        candidate = Path(path_text).expanduser().resolve()
        if not candidate.exists():
            raise FileNotFoundError(f"Benchmark directory not found: {candidate}")
        return candidate

    root = Path("./benchmark_results").resolve()
    if not root.exists():
        raise FileNotFoundError("No ./benchmark_results directory exists yet.")
    candidates = sorted([p for p in root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError("No benchmark run directories were found under ./benchmark_results.")
    return candidates[0]


def read_optional_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None


def read_optional_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return None


def pick_column(columns, candidates):
    cols = list(columns)
    for candidate in candidates:
        if candidate in cols:
            return candidate
    lowered = {str(col).strip().lower(): col for col in cols}
    for candidate in candidates:
        match = lowered.get(str(candidate).strip().lower())
        if match is not None:
            return match
    return None


def metric_column(metric_df, preferred="AUTO", metric_hint="rmse"):
    if metric_df is None or metric_df.empty:
        return None
    if str(preferred).strip() and str(preferred).strip().upper() != "AUTO":
        if preferred not in metric_df.columns:
            raise ValueError(f"Requested metric column not found: {preferred}")
        return preferred
    candidates = []
    hint = str(metric_hint).strip().lower()
    if hint == "rmse":
        candidates = [
            "test_rmse", "Test RMSE", "cv_rmse", "CV RMSE", "mean_test_rmse",
            "rmse", "RMSE"
        ]
    elif hint == "r2":
        candidates = ["test_r2", "Test R2", "cv_r2", "CV R2", "r2", "R2"]
    return pick_column(metric_df.columns, candidates)


def standardize_metric_frame(metric_df):
    if metric_df is None:
        return None
    frame = metric_df.copy()
    rename_map = {}
    dataset_col = pick_column(frame.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(frame.columns, ["model", "Model"])
    workflow_col = pick_column(frame.columns, ["workflow", "Workflow"])
    if dataset_col is not None:
        rename_map[dataset_col] = "dataset"
    if model_col is not None:
        rename_map[model_col] = "model"
    if workflow_col is not None:
        rename_map[workflow_col] = "workflow"
    frame = frame.rename(columns=rename_map)
    return frame


In [68]:
RUN_DIR = resolve_benchmark_dir(benchmark_run_dir)
SUMMARY_METRICS_FILE = standardize_metric_frame(read_optional_csv(RUN_DIR / "summary_metrics.csv"))
TEST_RMSE_PIVOT = read_optional_csv(RUN_DIR / "test_rmse_pivot.csv")
PREDICTIONS = read_optional_csv(RUN_DIR / "predictions.csv")
GA_HISTORY = read_optional_csv(RUN_DIR / "ga_history.csv")
RUN_CONFIG = read_optional_json(RUN_DIR / "run_config.json")
LEADERBOARD_TOP10 = read_optional_csv(RUN_DIR / "leaderboard_top10_reference.csv")
LEADERBOARD_COMPARISON = read_optional_csv(RUN_DIR / "leaderboard_comparison_by_dataset.csv")

run_status_rows = []
for status_path in sorted(RUN_DIR.glob("*/run_status.json")):
    payload = read_optional_json(status_path)
    if not isinstance(payload, dict):
        continue
    run_status_rows.append(
        {
            "dataset": status_path.parent.name,
            "status": payload.get("status", "unknown"),
            "checkpoint_stage": payload.get("checkpoint_stage", ""),
            "n_metrics_rows": payload.get("n_metrics_rows", np.nan),
            "started_at": payload.get("started_at", ""),
            "completed_at": payload.get("completed_at", ""),
            "elapsed_seconds": payload.get("elapsed_seconds", np.nan),
        }
    )
RUN_STATUS = pd.DataFrame(run_status_rows)

def _row_has_error_text_series(series):
    text = series.fillna("").astype(str).str.strip().str.lower()
    return text.ne("") & ~text.isin({"nan", "none"})

dataset_metric_tables = []
for metrics_path in sorted(RUN_DIR.glob("*/metrics.csv")):
    dataset_name = metrics_path.parent.name
    try:
        frame = pd.read_csv(metrics_path)
    except Exception:
        continue
    frame = standardize_metric_frame(frame)
    if frame is None or frame.empty:
        continue
    if "dataset" not in frame.columns:
        frame["dataset"] = dataset_name
    dataset_metric_tables.append(frame)

combined_metric_frames = []
if SUMMARY_METRICS_FILE is not None and not SUMMARY_METRICS_FILE.empty:
    summary_frame = SUMMARY_METRICS_FILE.copy()
    if "dataset" not in summary_frame.columns:
        raise ValueError("summary_metrics.csv is missing a dataset column.")
    summary_frame["_metrics_source"] = "summary_metrics_csv"
    combined_metric_frames.append(summary_frame)

if dataset_metric_tables:
    dataset_frame = pd.concat(dataset_metric_tables, ignore_index=True)
    dataset_frame["_metrics_source"] = "dataset_metrics"
    combined_metric_frames.append(dataset_frame)

if not combined_metric_frames:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")

combined_metrics = pd.concat(combined_metric_frames, ignore_index=True)
combined_metrics = standardize_metric_frame(combined_metrics)

if combined_metrics is None or combined_metrics.empty:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")
if "dataset" not in combined_metrics.columns:
    raise ValueError("Could not identify a dataset column in the metrics table.")
if "model" not in combined_metrics.columns:
    raise ValueError("Could not identify a model column in the metrics table.")

if "workflow" not in combined_metrics.columns:
    combined_metrics["workflow"] = ""

combined_metrics["_source_rank"] = combined_metrics.get("_metrics_source", "").map(
    {"dataset_metrics": 0, "summary_metrics_csv": 1}
).fillna(9)
combined_metrics["_error_rank"] = _row_has_error_text_series(combined_metrics.get("error", pd.Series(index=combined_metrics.index, dtype=object))).astype(int)
status_series = combined_metrics.get("status", pd.Series(index=combined_metrics.index, dtype=object)).fillna("").astype(str).str.strip().str.lower()
combined_metrics["_status_rank"] = np.where(status_series.str.startswith("skipped") | status_series.str.contains("error", regex=False), 1, 0)

_rmse_probe_col = metric_column(combined_metrics, rmse_metric_column_preference, metric_hint="rmse")
if _rmse_probe_col is not None:
    combined_metrics["_rmse_rank"] = pd.to_numeric(combined_metrics[_rmse_probe_col], errors="coerce")
else:
    combined_metrics["_rmse_rank"] = np.nan
combined_metrics["_rmse_rank"] = combined_metrics["_rmse_rank"].fillna(np.inf)

key_cols = ["dataset", "model", "workflow"]
combined_metrics = combined_metrics.sort_values(
    by=key_cols + ["_error_rank", "_status_rank", "_rmse_rank", "_source_rank"],
    ascending=[True, True, True, True, True, True, True],
)
SUMMARY_METRICS = combined_metrics.groupby(key_cols, as_index=False, dropna=False).first()
SUMMARY_METRICS = SUMMARY_METRICS.drop(columns=["_source_rank", "_error_rank", "_status_rank", "_rmse_rank", "_metrics_source"], errors="ignore")

if SUMMARY_METRICS is None or SUMMARY_METRICS.empty:
    raise ValueError(f"No metrics tables were found under {RUN_DIR}")

if "dataset" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a dataset column in the metrics table.")
if "model" not in SUMMARY_METRICS.columns:
    raise ValueError("Could not identify a model column in the metrics table.")

if {"model", "workflow"}.issubset(set(SUMMARY_METRICS.columns)):
    tabpfn_mask = SUMMARY_METRICS["model"].astype(str).str.contains("tabpfn", case=False, na=False)
    SUMMARY_METRICS.loc[tabpfn_mask, "workflow"] = "TabPFN deep learning"
    cfa_mask = SUMMARY_METRICS["model"].astype(str).str.startswith("CFA (", na=False)
    SUMMARY_METRICS.loc[cfa_mask, "workflow"] = "CFA fusion"

RMSE_COL = metric_column(SUMMARY_METRICS, rmse_metric_column_preference, metric_hint="rmse")
R2_COL = metric_column(SUMMARY_METRICS, "AUTO", metric_hint="r2")
SUMMARY_RMSE_PIVOT = SUMMARY_METRICS.pivot_table(index="dataset", columns="model", values=RMSE_COL, aggfunc="mean").reset_index()

print(f"Benchmark run directory: {RUN_DIR}")
print(f"Datasets: {SUMMARY_METRICS['dataset'].nunique()}")
print(f"Models: {SUMMARY_METRICS['model'].nunique()}")
print(f"Rows in merged summary table: {len(SUMMARY_METRICS):,}")
if SUMMARY_METRICS_FILE is not None:
    print(f"Rows in summary_metrics.csv: {len(SUMMARY_METRICS_FILE):,}")
print(f"Rows from dataset-level metrics after dedupe: {len(SUMMARY_METRICS):,}")
print(f"RMSE column used: {RMSE_COL}")
if R2_COL is not None:
    print(f"R2 column used: {R2_COL}")

if "model" in SUMMARY_METRICS.columns:
    tabpfn_rows = SUMMARY_METRICS[SUMMARY_METRICS["model"].astype(str).str.contains("TabPFNRegressor", na=False)]
    if not tabpfn_rows.empty:
        print(
            "TabPFNRegressor rows retained for analysis: "
            f"{len(tabpfn_rows)} rows across {tabpfn_rows['dataset'].nunique()} datasets."
        )

if RUN_CONFIG is not None:
    display_note("Loaded `run_config.json` for this benchmark run.")

if not RUN_STATUS.empty:
    completed = int((RUN_STATUS["status"].astype(str).str.lower() == "completed").sum())
    print(f"Datasets with run_status.json: {len(RUN_STATUS)} (completed: {completed})")

if LEADERBOARD_TOP10 is not None:
    print(f"Leaderboard top-10 reference rows: {len(LEADERBOARD_TOP10):,}")
if LEADERBOARD_COMPARISON is not None:
    print(f"Leaderboard comparison rows: {len(LEADERBOARD_COMPARISON):,}")


Benchmark run directory: C:\Users\Scott.Coffin\OneDrive - California OEHHA\R_new\AutoQSAR\benchmark_results\all_benchmarks_no_unimol_20260425_223505
Datasets: 1
Models: 12
Rows in merged summary table: 12
Rows from dataset-level metrics after dedupe: 12
RMSE column used: test_rmse
R2 column used: test_r2


Loaded `run_config.json` for this benchmark run.

Datasets with run_status.json: 1 (completed: 0)
Leaderboard top-10 reference rows: 144


In [69]:
config_summary = pd.DataFrame([RUN_CONFIG]) if isinstance(RUN_CONFIG, dict) else pd.DataFrame()
if not config_summary.empty:
    display(config_summary.T.rename(columns={0: "value"}))

best_by_dataset = SUMMARY_METRICS.sort_values(RMSE_COL, ascending=True).groupby("dataset", as_index=False).first()
best_by_dataset = best_by_dataset[[col for col in ["dataset", "workflow", "model", RMSE_COL, R2_COL] if col in best_by_dataset.columns]]
display_note("### Best model per dataset")
display(best_by_dataset.sort_values(RMSE_COL, ascending=True).reset_index(drop=True))

model_summary = (
    SUMMARY_METRICS.groupby([col for col in ["workflow", "model"] if col in SUMMARY_METRICS.columns], dropna=False)
    .agg(
        datasets=("dataset", "nunique"),
        mean_rmse=(RMSE_COL, "mean"),
        median_rmse=(RMSE_COL, "median"),
        std_rmse=(RMSE_COL, "std"),
        best_rmse=(RMSE_COL, "min"),
        worst_rmse=(RMSE_COL, "max"),
        mean_r2=(R2_COL, "mean") if R2_COL is not None else (RMSE_COL, "size"),
    )
    .reset_index()
)
if R2_COL is None and "mean_r2" in model_summary.columns:
    model_summary = model_summary.drop(columns=["mean_r2"])
display_note("### Cross-dataset model summary")
display(model_summary.sort_values("mean_rmse", ascending=True).head(int(show_top_n_models)).reset_index(drop=True))


,value
dataset,None
refresh_leaderboards_only,False
dataset_name,None
include_local_csv,None
output_dir,benchmark_results\all_benchmarks_no_unimol_202...
...,...
ga_models_resolved,
ga_resolution,"{'mode': 'disabled', 'reason': 'empty_ga_models'}"
default_feature_families,"[morgan, ecfp6, fcfp6, layered, atom_pair, top..."
config_signature,"{""benchmark_profile"": ""cost_optimized"", ""cfa_i..."


### Best model per dataset

,dataset,workflow,model,test_rmse,test_r2
0,tdc_carcinogens_lagunin,conventional,Random forest,0.338184,0.275435


### Cross-dataset model summary

,workflow,model,datasets,mean_rmse,median_rmse,std_rmse,best_rmse,worst_rmse,mean_r2
0,conventional,Random forest,1,0.338184,0.338184,NaN,0.338184,0.338184,0.275435
1,conventional,LogisticRegression,1,0.346022,0.346022,NaN,0.346022,0.346022,0.241462
2,conventional,CatBoost,1,0.350861,0.350861,NaN,0.350861,0.350861,0.220098
3,conventional,Extra trees,1,0.352601,0.352601,NaN,0.352601,0.352601,0.212342
4,conventional,"Voting Classifier (KNN, SVM)",1,0.354331,0.354331,NaN,0.354331,0.354331,0.204591
5,ChemML deep learning,ChemML MLP (PyTorch),1,0.358933,0.358933,NaN,0.358933,0.358933,0.183796
6,conventional,Tabular MLP,1,0.363796,0.363796,NaN,0.363796,0.363796,0.161532
7,TabPFN deep learning,TabPFNClassifier,1,0.368610,0.368610,NaN,0.368610,0.368610,0.139192
8,conventional,AdaBoost,1,0.369390,0.369390,NaN,0.369390,0.369390,0.135546
9,conventional,SVC,1,0.379682,0.379682,NaN,0.379682,0.379682,0.086704


### GA value diagnostics across runs

This section checks whether GA tuning actually improved benchmark performance in prior runs.

- It compares each GA-tuned model (for example, `CatBoost GA`) against its non-GA counterpart on the same dataset.
- Positive `delta_vs_baseline` means GA improved performance (direction-aware by metric type).
- This is computed across all run folders under `benchmark_results/`.
        


In [70]:
# GA_IMPACT_DIAGNOSTICS_MARKER
# Evaluate GA uplift by dataset/model and aggregate across prior benchmark runs.


def ga_baseline_candidates(ga_model_name):
    ga_model_name = str(ga_model_name or "").strip()
    candidates = []
    if ga_model_name.endswith(" GA"):
        base = ga_model_name[: -len(" GA")].strip()
        if base:
            candidates.extend([base, f"Tuned {base}"])
    elif ga_model_name.startswith("Tuned "):
        base = ga_model_name[len("Tuned ") :].strip()
        if base:
            candidates.extend([base])
    return [c for c in candidates if c]


def metric_is_lower_better(metric_name):
    text = str(metric_name or "").strip().lower()
    if not text:
        return True
    if any(key in text for key in ["rmse", "mae", "mse", "loss", "error"]):
        return True
    if any(key in text for key in ["r2", "spearman", "pearson", "auc", "accuracy", "f1"]):
        return False
    return True


def load_run_metric_table(run_dir):
    summary = standardize_metric_frame(read_optional_csv(run_dir / "summary_metrics.csv"))
    if summary is not None and not summary.empty:
        return summary

    parts = []
    for metrics_path in sorted(run_dir.glob("*/metrics.csv")):
        dataset_name = metrics_path.parent.name
        frame = standardize_metric_frame(pd.read_csv(metrics_path))
        if frame is None or frame.empty:
            continue
        if "dataset" not in frame.columns:
            frame["dataset"] = dataset_name
        parts.append(frame)
    if not parts:
        return None
    return pd.concat(parts, ignore_index=True)


benchmark_root = RUN_DIR.parent if RUN_DIR.parent.name == "benchmark_results" else Path("./benchmark_results").resolve()
run_dirs = sorted([p for p in benchmark_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)

ga_rows = []
ga_attempt_rows = []
for run_dir in run_dirs:
    run_name = run_dir.name
    metrics = load_run_metric_table(run_dir)
    if metrics is None or metrics.empty:
        continue

    metrics = metrics.copy()
    rmse_col = metric_column(metrics, "AUTO", metric_hint="rmse")

    # Normalize key metric columns.
    for col in ["primary_metric_value", "primary_metric", rmse_col, "dataset", "model", "workflow"]:
        if col is not None and col in metrics.columns:
            if col in {"primary_metric_value", rmse_col}:
                metrics[col] = pd.to_numeric(metrics[col], errors="coerce")
            else:
                metrics[col] = metrics[col].astype(str)

    model_series = metrics["model"].astype(str) if "model" in metrics.columns else pd.Series(dtype=str)
    workflow_series = metrics["workflow"].astype(str).str.lower() if "workflow" in metrics.columns else pd.Series(dtype=str)
    ga_mask = model_series.str.endswith(" GA") | workflow_series.str.contains("ga", regex=False)
    ga_metrics = metrics.loc[ga_mask].copy()
    if ga_metrics.empty:
        continue

    for _, ga_row in ga_metrics.iterrows():
        dataset_name = str(ga_row.get("dataset", ""))
        ga_model_name = str(ga_row.get("model", "")).strip()
        ga_error_text = str(ga_row.get("error", "")).strip() if "error" in ga_row.index else ""
        ga_attempt_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "ga_model": ga_model_name,
                "workflow": str(ga_row.get("workflow", "")),
                "error": ga_error_text,
                "has_error": bool(ga_error_text),
                "has_primary_metric": bool(pd.notna(ga_row.get("primary_metric_value", np.nan))),
            }
        )
        candidates = ga_baseline_candidates(ga_model_name)
        if not candidates:
            continue

        dataset_rows = metrics.loc[metrics["dataset"].astype(str) == dataset_name].copy()
        baseline_rows = dataset_rows.loc[dataset_rows["model"].astype(str).isin(candidates)].copy()
        baseline_rows = baseline_rows.loc[~baseline_rows["model"].astype(str).str.endswith(" GA")]
        if baseline_rows.empty:
            continue

        baseline = baseline_rows.iloc[0]

        ga_primary_name = str(ga_row.get("primary_metric", "")).strip().lower()
        baseline_primary_name = str(baseline.get("primary_metric", "")).strip().lower()
        use_primary = (
            "primary_metric_value" in ga_row.index
            and "primary_metric_value" in baseline.index
            and pd.notna(ga_row.get("primary_metric_value", np.nan))
            and pd.notna(baseline.get("primary_metric_value", np.nan))
            and ga_primary_name
            and baseline_primary_name
            and ga_primary_name == baseline_primary_name
        )

        if use_primary:
            metric_name = ga_primary_name
            ga_value = float(ga_row.get("primary_metric_value", np.nan))
            base_value = float(baseline.get("primary_metric_value", np.nan))
        elif rmse_col is not None and rmse_col in ga_row.index and rmse_col in baseline.index:
            metric_name = "rmse"
            ga_value = float(ga_row.get(rmse_col, np.nan))
            base_value = float(baseline.get(rmse_col, np.nan))
        else:
            continue

        if not np.isfinite(ga_value) or not np.isfinite(base_value):
            continue

        lower_better = metric_is_lower_better(metric_name)
        delta = (base_value - ga_value) if lower_better else (ga_value - base_value)
        denom = abs(base_value) if abs(base_value) > 1e-12 else np.nan
        rel_delta = (delta / denom) if np.isfinite(denom) else np.nan

        ga_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "ga_model": ga_model_name,
                "baseline_model": str(baseline.get("model", "")),
                "metric": metric_name,
                "lower_is_better": bool(lower_better),
                "ga_value": float(ga_value),
                "baseline_value": float(base_value),
                "delta_vs_baseline": float(delta),
                "relative_delta": float(rel_delta) if pd.notna(rel_delta) else np.nan,
                "ga_better": bool(delta > 0.0),
            }
        )

GA_IMPACT = pd.DataFrame(ga_rows)
GA_ATTEMPTS = pd.DataFrame(ga_attempt_rows)

if GA_IMPACT.empty:
    if GA_ATTEMPTS.empty:
        display_note("No GA model rows were found in the scanned run directories.")
    else:
        display_note("No comparable GA-vs-baseline pairs with valid metrics were found. GA attempt diagnostics are shown below.")
        attempt_summary = (
            GA_ATTEMPTS.groupby("run_name", dropna=False)
            .agg(
                ga_rows=("ga_model", "size"),
                ga_rows_with_error=("has_error", "sum"),
                ga_rows_with_primary_metric=("has_primary_metric", "sum"),
            )
            .reset_index()
        )
        display(attempt_summary.sort_values("run_name").reset_index(drop=True))

        error_rows = GA_ATTEMPTS.loc[GA_ATTEMPTS["has_error"]].copy()
        if not error_rows.empty:
            error_summary = (
                error_rows.groupby(["run_name", "ga_model", "error"], dropna=False)
                .size()
                .reset_index(name="count")
                .sort_values("count", ascending=False)
            )
            display_note("Top GA error signatures")
            display(error_summary.head(20).reset_index(drop=True))
else:
    display_note("### GA uplift by dataset")
    display(
        GA_IMPACT.sort_values(["ga_better", "delta_vs_baseline"], ascending=[False, False])
        .reset_index(drop=True)
    )

    ga_summary = (
        GA_IMPACT.groupby(["run_name", "ga_model"], dropna=False)
        .agg(
            datasets_compared=("dataset", "nunique"),
            wins=("ga_better", "sum"),
            mean_delta=("delta_vs_baseline", "mean"),
            median_delta=("delta_vs_baseline", "median"),
            mean_relative_delta=("relative_delta", "mean"),
        )
        .reset_index()
    )
    ga_summary["win_rate"] = ga_summary["wins"] / ga_summary["datasets_compared"].replace(0, np.nan)

    display_note("### GA uplift summary by run and GA model")
    display(ga_summary.sort_values(["win_rate", "mean_delta"], ascending=[False, False]).reset_index(drop=True))

    any_wins = GA_IMPACT.loc[GA_IMPACT["ga_better"]].copy()
    if any_wins.empty:
        display_note("GA did not beat its non-GA counterpart on any comparable dataset in the scanned runs.")
    else:
        best_wins = any_wins.sort_values("delta_vs_baseline", ascending=False).head(25)
        display_note("### Datasets where GA improved performance")
        display(best_wins.reset_index(drop=True))

    fig = px.bar(
        ga_summary.sort_values("win_rate", ascending=False),
        x="ga_model",
        y="win_rate",
        color="run_name",
        barmode="group",
        hover_data=[c for c in ["datasets_compared", "wins", "mean_delta", "mean_relative_delta"] if c in ga_summary.columns],
        title="GA win rate vs non-GA baseline",
    )
    fig.update_layout(xaxis_title="GA model", yaxis_title="Win rate")
    fig.show()
        


No comparable GA-vs-baseline pairs with valid metrics were found. GA attempt diagnostics are shown below.

,run_name,ga_rows,ga_rows_with_error,ga_rows_with_primary_metric
0,all_benchmarks_full_no_unimol_20260418_095237,24,24,0


Top GA error signatures

,run_name,ga_model,error,count
0,all_benchmarks_full_no_unimol_20260418_095237,CatBoost GA,'list' object has no attribute 'split',12
1,all_benchmarks_full_no_unimol_20260418_095237,ElasticNet GA,'list' object has no attribute 'split',12


In [71]:
# Architecture coverage and prediction-diversity diagnostics


def classify_architecture_family(workflow_name, model_name):
    workflow_text = str(workflow_name or "").strip().lower()
    model_text = str(model_name or "").strip().lower()

    if "ensemble" in workflow_text or model_text.startswith("ensemble ("):
        return "Ensemble meta-model"
    if "cfa" in workflow_text or model_text.startswith("cfa ("):
        return "CFA combinatorial fusion"
    if "tabpfn" in workflow_text or "tabpfn" in model_text:
        return "TabPFN foundation model"
    if "chemprop" in workflow_text or "chemprop" in model_text:
        return "Graph neural networks (Chemprop)"
    if "maplight + gnn" in workflow_text or "maplight + gnn" in model_text:
        return "Graph transfer + tabular head"
    if "uni-mol" in workflow_text or "uni-mol" in model_text or "unimol" in model_text:
        return "3D foundation model"
    if "chemml deep learning" in workflow_text or "chemml" in model_text:
        return "Deep tabular neural nets"
    if "tuned conventional" in workflow_text:
        return "Tuned conventional ML"
    if "conventional" in workflow_text:
        return "Conventional ML"
    return "Other / unknown"


def classify_architecture_subfamily(model_name):
    model_text = str(model_name or "").strip().lower()
    if "cmpnn" in model_text:
        return "CMPNN"
    if "attentivefp" in model_text:
        return "AttentiveFP"
    if "d-mpnn" in model_text or "dmpnn" in model_text:
        return "D-MPNN"
    if "maplight + gnn" in model_text:
        return "MapLight + GNN"
    if model_text.startswith("cfa (") or "combinatorial fusion" in model_text:
        return "CFA fusion"
    if "tabpfn" in model_text:
        return "TabPFN"
    if "chemml mlp" in model_text:
        return "ChemML MLP"
    if "catboost" in model_text:
        return "CatBoost-family"
    if "xgboost" in model_text:
        return "XGBoost-family"
    if "random forest" in model_text:
        return "RandomForest-family"
    if "svr" in model_text:
        return "SVR"
    if "elasticnet" in model_text:
        return "ElasticNet-family"
    if model_text.startswith("ensemble ("):
        return "Stacking / averaging"
    return str(model_name or "")


coverage_source = SUMMARY_METRICS.copy()
error_col = pick_column(coverage_source.columns, ["error", "Error"])
if error_col is not None:
    coverage_source = coverage_source.loc[
        coverage_source[error_col].isna()
        | (coverage_source[error_col].astype(str).str.strip() == "")
    ].copy()

if coverage_source.empty:
    display_note("No successful model rows were available for architecture-coverage diagnostics.")
else:
    coverage_source["architecture_family"] = coverage_source.apply(
        lambda row: classify_architecture_family(row.get("workflow", ""), row.get("model", "")),
        axis=1,
    )
    coverage_source["architecture_subfamily"] = coverage_source["model"].apply(classify_architecture_subfamily)

    family_summary = (
        coverage_source.groupby("architecture_family", dropna=False)
        .agg(
            datasets=("dataset", "nunique"),
            models=("model", "nunique"),
            rows=("model", "size"),
            mean_rmse=(RMSE_COL, "mean") if RMSE_COL in coverage_source.columns else ("model", "size"),
        )
        .reset_index()
        .sort_values(["datasets", "rows"], ascending=[False, False])
    )

    display_note("### Architecture Coverage")
    display(family_summary)

    dataset_family_presence = (
        coverage_source.assign(present=1)
        .groupby(["dataset", "architecture_family"], dropna=False)["present"]
        .max()
        .unstack(fill_value=0)
        .sort_index()
    )

    coverage_fraction = dataset_family_presence.mean(axis=1).rename("coverage_fraction").reset_index()
    coverage_fraction = coverage_fraction.sort_values("coverage_fraction", ascending=False)
    display_note("### Per-dataset architecture-family coverage")
    display(coverage_fraction)

    coverage_fig = px.imshow(
        dataset_family_presence,
        aspect="auto",
        color_continuous_scale=[[0.0, "#f0f0f0"], [1.0, "#2ca25f"]],
        labels={"x": "Architecture family", "y": "Dataset", "color": "Present"},
        title="Architecture-family coverage matrix (dataset x family)",
        text_auto=True,
    )
    coverage_fig.update_layout(height=max(420, 45 * len(dataset_family_presence.index)))
    coverage_fig.show()

    if PREDICTIONS is None or PREDICTIONS.empty:
        display_note("No predictions table was found; skipping architecture-diversity diagnostics.")
    else:
        pred_df = PREDICTIONS.copy()
        dataset_col = pick_column(pred_df.columns, ["dataset", "dataset_id", "dataset_name"])
        model_col = pick_column(pred_df.columns, ["model", "Model"])
        predicted_col = pick_column(pred_df.columns, ["predicted", "prediction", "y_pred"])
        split_col = pick_column(pred_df.columns, ["split", "Split"])
        smiles_col = pick_column(pred_df.columns, ["smiles", "SMILES", "canonical_smiles"])
        row_col = pick_column(pred_df.columns, ["row_id", "Row", "index"])

        required = [dataset_col, model_col, predicted_col]
        if any(col is None for col in required):
            display_note("Predictions table is missing dataset/model/prediction columns; skipping diversity diagnostics.")
        else:
            pred_df = pred_df.rename(columns={dataset_col: "dataset", model_col: "model", predicted_col: "predicted"})
            if split_col is not None:
                pred_df = pred_df.rename(columns={split_col: "split"})
                pred_df = pred_df.loc[pred_df["split"].astype(str).str.lower() == "test"].copy()

            if smiles_col is not None:
                pred_df = pred_df.rename(columns={smiles_col: "smiles_key"})
            elif row_col is not None:
                pred_df = pred_df.rename(columns={row_col: "smiles_key"})
            else:
                pred_df["smiles_key"] = pred_df.groupby(["dataset", "model"]).cumcount().astype(str)

            pred_df["predicted"] = pd.to_numeric(pred_df["predicted"], errors="coerce")
            pred_df = pred_df.loc[pred_df["predicted"].notna()].copy()

            diversity_rows = []
            pair_rows = []
            for dataset_name, group in pred_df.groupby("dataset", sort=True):
                wide = group.pivot_table(index="smiles_key", columns="model", values="predicted", aggfunc="mean")
                if wide.shape[1] < 2:
                    continue

                corr = wide.corr(method="spearman")
                if corr.shape[0] < 2:
                    continue

                corr_values = corr.to_numpy(dtype=float)
                tri = np.triu_indices_from(corr_values, k=1)
                pair_corr = corr_values[tri]
                pair_corr = pair_corr[np.isfinite(pair_corr)]
                if pair_corr.size == 0:
                    continue

                abs_corr = np.abs(pair_corr)
                diversity_rows.append(
                    {
                        "dataset": dataset_name,
                        "models_in_pairwise_corr": int(corr.shape[0]),
                        "pair_count": int(pair_corr.size),
                        "mean_abs_spearman": float(np.mean(abs_corr)),
                        "median_abs_spearman": float(np.median(abs_corr)),
                        "max_abs_spearman": float(np.max(abs_corr)),
                        "diversity_index": float(1.0 - np.median(abs_corr)),
                    }
                )

                cols = list(corr.columns)
                for i in range(len(cols)):
                    for j in range(i + 1, len(cols)):
                        value = corr.iat[i, j]
                        if pd.isna(value):
                            continue
                        pair_rows.append(
                            {
                                "dataset": dataset_name,
                                "model_a": str(cols[i]),
                                "model_b": str(cols[j]),
                                "spearman": float(value),
                                "abs_spearman": float(abs(value)),
                            }
                        )

            if not diversity_rows:
                display_note("Prediction diversity diagnostics could not be computed (insufficient aligned test predictions).")
            else:
                diversity_df = pd.DataFrame(diversity_rows).sort_values("diversity_index", ascending=False).reset_index(drop=True)
                display_note("### Prediction Diversity Across Architectures")
                display(diversity_df)

                diversity_fig = px.bar(
                    diversity_df.sort_values("diversity_index", ascending=True),
                    x="dataset",
                    y="diversity_index",
                    hover_data=["models_in_pairwise_corr", "median_abs_spearman", "max_abs_spearman"],
                    title="Architecture diversity index by dataset (1 - median |Spearman|)",
                )
                diversity_fig.update_layout(yaxis_title="Diversity index (higher is better)", xaxis_title="Dataset")
                diversity_fig.show()

                if pair_rows:
                    pair_df = pd.DataFrame(pair_rows)
                    pair_df["pair"] = pair_df.apply(
                        lambda row: " / ".join(sorted([str(row["model_a"]), str(row["model_b"])])),
                        axis=1,
                    )
                    pair_summary = (
                        pair_df.groupby("pair", dropna=False)
                        .agg(
                            datasets=("dataset", "nunique"),
                            mean_abs_spearman=("abs_spearman", "mean"),
                            median_abs_spearman=("abs_spearman", "median"),
                            max_abs_spearman=("abs_spearman", "max"),
                        )
                        .reset_index()
                        .sort_values(["mean_abs_spearman", "datasets"], ascending=[False, False])
                    )
                    display_note("### Most redundant model pairs (cross-dataset)")
                    display(pair_summary.head(20))



### Architecture Coverage

,architecture_family,datasets,models,rows,mean_rmse
0,Conventional ML,1,10,10,0.364936
1,Deep tabular neural nets,1,1,1,0.358933
2,TabPFN foundation model,1,1,1,0.368610


### Per-dataset architecture-family coverage

,dataset,coverage_fraction
0,tdc_carcinogens_lagunin,1.0


No predictions table was found; skipping architecture-diversity diagnostics.

In [72]:
# Feature-family representation analysis from selector outputs

feature_family_drop_presence_threshold = 0.20
feature_family_drop_enrichment_threshold = 0.25
feature_family_drop_selected_total_threshold = 25
feature_family_watch_enrichment_threshold = 0.35


def feature_family_from_name(feature_name):
    name = str(feature_name or "")
    if name.startswith("morgan_bit_"):
        return "morgan"
    if name.startswith("ecfp6_bit_"):
        return "ecfp6"
    if name.startswith("fcfp6_bit_"):
        return "fcfp6"
    if name.startswith("layered_bit_"):
        return "layered"
    if name.startswith("atom_pair_bit_"):
        return "atom_pair"
    if name.startswith("topological_torsion_bit_"):
        return "topological_torsion"
    if name.startswith("rdk_path_bit_"):
        return "rdk_path"
    if name.startswith("maccs_bit_"):
        return "maccs"
    if name.startswith("rdkit_"):
        return "rdkit"
    if (
        name.startswith("maplight_morgan_")
        or name.startswith("avalon_count_")
        or name.startswith("erg_")
        or name.startswith("maplight_desc_")
    ):
        return "maplight"
    return "other"


def infer_feature_col(frame):
    return pick_column(frame.columns, ["feature", "Feature", "name"])


def infer_magnitude_col(frame):
    if "abs_coefficient" in frame.columns:
        return "abs_coefficient"
    if "importance" in frame.columns:
        return "importance"
    if "coefficient" in frame.columns:
        return "coefficient"
    return None


selector_rows = []
missing_selector_artifacts = []
for dataset_dir in sorted([p for p in RUN_DIR.iterdir() if p.is_dir()]):
    dataset_name = dataset_dir.name
    selected_path = dataset_dir / "selected_features.csv"
    coeff_path = dataset_dir / "selector_coefficients.csv"
    if not selected_path.exists() or not coeff_path.exists():
        missing_selector_artifacts.append(dataset_name)
        continue

    try:
        selected_df = pd.read_csv(selected_path)
        coeff_df = pd.read_csv(coeff_path)
    except Exception as exc:
        missing_selector_artifacts.append(f"{dataset_name} ({type(exc).__name__})")
        continue

    feature_col = infer_feature_col(coeff_df)
    if feature_col is None:
        missing_selector_artifacts.append(f"{dataset_name} (no feature column)")
        continue

    selected_feature_col = infer_feature_col(selected_df)
    selected_set = set()
    if selected_feature_col is not None:
        selected_set = set(selected_df[selected_feature_col].astype(str).tolist())

    coeff_work = coeff_df.copy()
    coeff_work["feature"] = coeff_work[feature_col].astype(str)
    coeff_work["feature_family"] = coeff_work["feature"].map(feature_family_from_name)
    coeff_work["is_selected"] = coeff_work["feature"].isin(selected_set)

    magnitude_col = infer_magnitude_col(coeff_work)
    if magnitude_col is None:
        coeff_work["feature_magnitude"] = np.nan
    elif magnitude_col == "coefficient":
        coeff_work["feature_magnitude"] = pd.to_numeric(coeff_work[magnitude_col], errors="coerce").abs()
    else:
        coeff_work["feature_magnitude"] = pd.to_numeric(coeff_work[magnitude_col], errors="coerce")

    grouped = coeff_work.groupby("feature_family", dropna=False)
    for family_name, family_df in grouped:
        selected_mask = family_df["is_selected"].astype(bool)
        selector_rows.append(
            {
                "dataset": dataset_name,
                "feature_family": str(family_name),
                "available_features": int(len(family_df)),
                "selected_features": int(selected_mask.sum()),
                "selected_magnitude_sum": float(
                    family_df.loc[selected_mask, "feature_magnitude"].fillna(0.0).sum()
                ),
            }
        )

if not selector_rows:
    display_note("No selector feature artifacts were found in this run; skipping feature-family representation analysis.")
else:
    selector_family_df = pd.DataFrame(selector_rows)

    family_summary = (
        selector_family_df.groupby("feature_family", dropna=False)
        .agg(
            datasets_with_family=("dataset", "nunique"),
            datasets_selected=("selected_features", lambda s: int((pd.to_numeric(s, errors="coerce").fillna(0) > 0).sum())),
            available_features_total=("available_features", "sum"),
            selected_features_total=("selected_features", "sum"),
            mean_selected_per_dataset=("selected_features", "mean"),
            median_selected_per_dataset=("selected_features", "median"),
            selected_magnitude_total=("selected_magnitude_sum", "sum"),
        )
        .reset_index()
    )

    family_summary["dataset_presence_rate"] = family_summary["datasets_selected"] / family_summary["datasets_with_family"].replace(0, np.nan)
    family_summary["selection_rate"] = family_summary["selected_features_total"] / family_summary["available_features_total"].replace(0, np.nan)

    total_available = float(family_summary["available_features_total"].sum())
    total_selected = float(family_summary["selected_features_total"].sum())
    total_magnitude = float(family_summary["selected_magnitude_total"].sum())

    if total_available > 0 and total_selected > 0:
        family_summary["expected_selected_if_uniform"] = family_summary["available_features_total"] / total_available * total_selected
        family_summary["enrichment_vs_uniform"] = family_summary["selected_features_total"] / family_summary["expected_selected_if_uniform"].replace(0, np.nan)
    else:
        family_summary["expected_selected_if_uniform"] = np.nan
        family_summary["enrichment_vs_uniform"] = np.nan

    family_summary["selected_share"] = family_summary["selected_features_total"] / max(total_selected, 1.0)
    family_summary["selected_magnitude_share"] = family_summary["selected_magnitude_total"] / max(total_magnitude, 1.0)

    family_summary = family_summary.sort_values(["selected_features_total", "selection_rate"], ascending=[False, False]).reset_index(drop=True)

    display_note("### Feature-Family Representation In Selected Features")
    display(family_summary)

    selected_heatmap = (
        selector_family_df.pivot_table(
            index="dataset",
            columns="feature_family",
            values="selected_features",
            aggfunc="sum",
            fill_value=0,
        )
        .sort_index()
    )

    fig = px.imshow(
        selected_heatmap,
        aspect="auto",
        color_continuous_scale="Viridis",
        labels={"x": "Feature family", "y": "Dataset", "color": "# selected"},
        title="Selected feature counts by dataset and family",
        text_auto=True,
    )
    fig.update_layout(height=max(420, 45 * len(selected_heatmap.index)))
    fig.show()

    enrich_plot_df = family_summary.sort_values("enrichment_vs_uniform", ascending=True)
    fig = px.bar(
        enrich_plot_df,
        x="feature_family",
        y="enrichment_vs_uniform",
        hover_data=["selected_features_total", "available_features_total", "dataset_presence_rate", "selection_rate"],
        title="Feature-family selection enrichment vs uniform-by-dimension baseline",
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="black")
    fig.update_layout(xaxis_title="Feature family", yaxis_title="Enrichment (>1 means over-represented)")
    fig.show()

    drop_candidates = family_summary.loc[
        (family_summary["dataset_presence_rate"].fillna(0.0) <= float(feature_family_drop_presence_threshold))
        & (family_summary["enrichment_vs_uniform"].fillna(np.inf) <= float(feature_family_drop_enrichment_threshold))
        & (family_summary["selected_features_total"].fillna(0.0) <= float(feature_family_drop_selected_total_threshold))
    ].copy()

    watchlist = family_summary.loc[
        (family_summary["enrichment_vs_uniform"].fillna(np.inf) <= float(feature_family_watch_enrichment_threshold))
        & (family_summary["dataset_presence_rate"].fillna(0.0) >= 0.5)
    ].copy()

    if drop_candidates.empty:
        display_note(
            "No feature family met the strict drop-candidate rule in this run. "
            "That means no family looked both rare across datasets and consistently unselected."
        )
    else:
        display_note("### Suggested drop candidates (strict rule)")
        display(
            drop_candidates[[
                "feature_family",
                "datasets_selected",
                "datasets_with_family",
                "dataset_presence_rate",
                "selected_features_total",
                "available_features_total",
                "selection_rate",
                "enrichment_vs_uniform",
            ]].sort_values(["dataset_presence_rate", "selected_features_total"], ascending=[True, True]).reset_index(drop=True)
        )

    if not watchlist.empty:
        display_note("### Watchlist: under-enriched but still used across many datasets")
        display(
            watchlist[[
                "feature_family",
                "datasets_selected",
                "datasets_with_family",
                "dataset_presence_rate",
                "selected_features_total",
                "available_features_total",
                "selection_rate",
                "enrichment_vs_uniform",
            ]].sort_values("enrichment_vs_uniform", ascending=True).reset_index(drop=True)
        )

    if missing_selector_artifacts:
        display_note(
            "Selector artifacts were missing for some datasets and were excluded from this section: "
            + ", ".join(sorted(set(missing_selector_artifacts)))
        )




### Feature-Family Representation In Selected Features

,feature_family,datasets_with_family,datasets_selected,available_features_total,selected_features_total,mean_selected_per_dataset,median_selected_per_dataset,selected_magnitude_total,dataset_presence_rate,selection_rate,expected_selected_if_uniform,enrichment_vs_uniform,selected_share,selected_magnitude_share
0,maplight,1,1,1732,15,15.0,15.0,0.095087,1.0,0.008661,17.056953,0.879407,0.180723,0.095087
1,fcfp6,1,1,953,14,14.0,14.0,0.152409,1.0,0.014690,9.385263,1.491700,0.168675,0.152409
2,morgan,1,1,903,12,12.0,12.0,0.146913,1.0,0.013289,8.892857,1.349398,0.144578,0.146913
3,layered,1,1,1000,11,11.0,11.0,0.099234,1.0,0.011000,9.848125,1.116964,0.132530,0.099234
4,rdk_path,1,1,1024,8,8.0,8.0,0.069737,1.0,0.007812,10.084480,0.793298,0.096386,0.069737
5,maccs,1,1,131,7,7.0,7.0,0.115773,1.0,0.053435,1.290104,5.425917,0.084337,0.115773
6,topological_torsion,1,1,564,6,6.0,6.0,0.102075,1.0,0.010638,5.554343,1.080236,0.072289,0.102075
7,ecfp6,1,1,967,5,5.0,5.0,0.029307,1.0,0.005171,9.523137,0.525037,0.060241,0.029307
8,atom_pair,1,1,973,4,4.0,4.0,0.015573,1.0,0.004111,9.582226,0.417440,0.048193,0.015573
9,rdkit,1,1,181,1,1.0,1.0,0.013919,1.0,0.005525,1.782511,0.561006,0.012048,0.013919


No feature family met the strict drop-candidate rule in this run. That means no family looked both rare across datasets and consistently unselected.

In [73]:
# Leaderboard-relative performance summary (using cached run artifacts)

def leaderboard_metric_kind(text):
    t = str(text).strip().lower()
    if not t:
        return None
    if "mean_squared_error" in t or t == "mse" or " mse" in t:
        return "mse"
    if "rmse" in t:
        return "rmse"
    if "mae" in t:
        return "mae"
    if "spearman" in t:
        return "spearman"
    if "pearson" in t:
        return "pearson"
    if t in {"r2", "r^2", "r?"} or "r2" in t or "r^2" in t or "r?" in t:
        return "r2"
    return t


def leaderboard_direction(metric_kind):
    if metric_kind in {"mse", "rmse", "mae"}:
        return "lower"
    if metric_kind in {"r2", "spearman", "pearson"}:
        return "higher"
    return "lower"


def model_metric_column(frame, metric_kind):
    if metric_kind in {"mse", "rmse"}:
        return metric_column(frame, "AUTO", metric_hint="rmse")
    if metric_kind == "r2":
        return metric_column(frame, "AUTO", metric_hint="r2")
    if metric_kind == "mae":
        return pick_column(frame.columns, ["test_mae", "Test MAE", "cv_mae", "CV MAE", "mae", "MAE"])
    if metric_kind == "spearman":
        return pick_column(frame.columns, ["test_spearman", "Test Spearman", "cv_spearman", "CV Spearman", "spearman", "Spearman"])
    if metric_kind == "pearson":
        return pick_column(frame.columns, ["test_pearson", "Test Pearson", "cv_pearson", "CV Pearson", "pearson", "Pearson"])
    return None


def parse_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def to_leaderboard_scale(value, metric_kind):
    value = float(value)
    if metric_kind == "mse":
        return value ** 2
    return value


leaderboard_eval = None
if LEADERBOARD_COMPARISON is not None and not LEADERBOARD_COMPARISON.empty:
    leaderboard_eval = LEADERBOARD_COMPARISON.copy()

    rename_map = {
        "leaderboard_rank": "estimated_rank_vs_top10",
        "leaderboard_estimated_rank_vs_top10": "estimated_rank_vs_top10",
        "leaderboard_estimated_in_top10": "in_top10_estimate",
        "leaderboard_gap_to_top10_cutoff": "gap_to_top10_cutoff",
        "leaderboard_delta_primary": "gap_to_top1",
        "primary_metric_value": "best_value_raw_model_metric",
        "primary_metric": "best_value_metric_column",
    }
    applicable_renames = {k: v for k, v in rename_map.items() if k in leaderboard_eval.columns and v not in leaderboard_eval.columns}
    if applicable_renames:
        leaderboard_eval = leaderboard_eval.rename(columns=applicable_renames)

    if "best_value" not in leaderboard_eval.columns and "best_value_raw_model_metric" in leaderboard_eval.columns:
        leaderboard_eval["best_value"] = pd.to_numeric(leaderboard_eval["best_value_raw_model_metric"], errors="coerce")

    if "metric_kind" not in leaderboard_eval.columns:
        metric_source = None
        if "leaderboard_metric_name" in leaderboard_eval.columns:
            metric_source = leaderboard_eval["leaderboard_metric_name"].astype(str)
        elif "best_value_metric_column" in leaderboard_eval.columns:
            metric_source = leaderboard_eval["best_value_metric_column"].astype(str)
        if metric_source is not None:
            leaderboard_eval["metric_kind"] = metric_source.apply(leaderboard_metric_kind)

    if "direction" not in leaderboard_eval.columns and "metric_kind" in leaderboard_eval.columns:
        leaderboard_eval["direction"] = leaderboard_eval["metric_kind"].apply(leaderboard_direction)

    if "leaderboard_top1" not in leaderboard_eval.columns and "leaderboard_metric_reference" in leaderboard_eval.columns:
        leaderboard_eval["leaderboard_top1"] = pd.to_numeric(leaderboard_eval["leaderboard_metric_reference"], errors="coerce")

    if "leaderboard_top10_cutoff" not in leaderboard_eval.columns:
        leaderboard_eval["leaderboard_top10_cutoff"] = np.nan

    if all(col in leaderboard_eval.columns for col in ["best_value", "gap_to_top10_cutoff", "direction"]):
        best_vals = pd.to_numeric(leaderboard_eval["best_value"], errors="coerce")
        gap_vals = pd.to_numeric(leaderboard_eval["gap_to_top10_cutoff"], errors="coerce")
        direction_vals = leaderboard_eval["direction"].astype(str)
        cutoff_vals = pd.to_numeric(leaderboard_eval["leaderboard_top10_cutoff"], errors="coerce")
        missing_cutoff = cutoff_vals.isna() & best_vals.notna() & gap_vals.notna()
        lower_mask = missing_cutoff & direction_vals.eq("lower")
        higher_mask = missing_cutoff & direction_vals.eq("higher")
        leaderboard_eval.loc[lower_mask, "leaderboard_top10_cutoff"] = (best_vals - gap_vals)[lower_mask]
        leaderboard_eval.loc[higher_mask, "leaderboard_top10_cutoff"] = (best_vals + gap_vals)[higher_mask]

    if "gap_to_top1" not in leaderboard_eval.columns:
        leaderboard_eval["gap_to_top1"] = np.nan

    if all(col in leaderboard_eval.columns for col in ["best_value", "leaderboard_top1", "direction"]):
        best_vals = pd.to_numeric(leaderboard_eval["best_value"], errors="coerce")
        top1_vals = pd.to_numeric(leaderboard_eval["leaderboard_top1"], errors="coerce")
        direction_vals = leaderboard_eval["direction"].astype(str)
        gap_top1_vals = pd.to_numeric(leaderboard_eval["gap_to_top1"], errors="coerce")
        missing_gap = gap_top1_vals.isna() & best_vals.notna() & top1_vals.notna()
        lower_mask = missing_gap & direction_vals.eq("lower")
        higher_mask = missing_gap & direction_vals.eq("higher")
        leaderboard_eval.loc[lower_mask, "gap_to_top1"] = (best_vals - top1_vals)[lower_mask]
        leaderboard_eval.loc[higher_mask, "gap_to_top1"] = (top1_vals - best_vals)[higher_mask]

    if "in_top10_estimate" not in leaderboard_eval.columns and "estimated_rank_vs_top10" in leaderboard_eval.columns:
        rank_vals = pd.to_numeric(leaderboard_eval["estimated_rank_vs_top10"], errors="coerce")
        leaderboard_eval["in_top10_estimate"] = rank_vals <= 10

else:
    ref = LEADERBOARD_TOP10.copy() if LEADERBOARD_TOP10 is not None else None
    if ref is not None and not ref.empty and "is_top10_entry" in ref.columns:
        ref = ref.loc[ref["is_top10_entry"].fillna(False)].copy()

    rows = []
    if ref is not None and not ref.empty:
        for dataset_name, dataset_frame in SUMMARY_METRICS.groupby("dataset", sort=True):
            dataset_frame = dataset_frame.copy()

            benchmark_id = ""
            if "benchmark_id" in dataset_frame.columns:
                non_empty_ids = dataset_frame["benchmark_id"].dropna().astype(str)
                non_empty_ids = non_empty_ids[non_empty_ids.str.strip() != ""]
                if not non_empty_ids.empty:
                    benchmark_id = non_empty_ids.iloc[0]

            ref_subset = pd.DataFrame()
            if benchmark_id:
                ref_subset = ref.loc[ref["benchmark_id"].astype(str) == str(benchmark_id)].copy() if "benchmark_id" in ref.columns else pd.DataFrame()
            if ref_subset.empty and "dataset" in ref.columns:
                ref_subset = ref.loc[ref["dataset"].astype(str) == str(dataset_name)].copy()
            if ref_subset.empty:
                continue

            metric_name = str(ref_subset.get("leaderboard_metric_name", pd.Series([""])).dropna().iloc[0]) if "leaderboard_metric_name" in ref_subset.columns else ""
            metric_kind = leaderboard_metric_kind(metric_name)
            metric_col = model_metric_column(dataset_frame, metric_kind)
            if metric_col is None or metric_col not in dataset_frame.columns:
                continue

            candidate = dataset_frame.copy()
            if "error" in candidate.columns:
                candidate = candidate.loc[candidate["error"].isna() | (candidate["error"].astype(str).str.strip() == "")].copy()
            candidate[metric_col] = parse_numeric(candidate[metric_col])
            candidate = candidate.loc[candidate[metric_col].notna()].copy()
            if candidate.empty:
                continue
            candidate["_leaderboard_scale_value"] = candidate[metric_col].apply(lambda v: to_leaderboard_scale(v, metric_kind))

            direction = leaderboard_direction(metric_kind)
            if direction == "lower":
                best_idx = candidate["_leaderboard_scale_value"].idxmin()
            else:
                best_idx = candidate["_leaderboard_scale_value"].idxmax()
            best_row = candidate.loc[best_idx]
            best_value_raw = float(best_row[metric_col])
            best_value = float(best_row["_leaderboard_scale_value"])

            metric_values = parse_numeric(ref_subset.get("metric_value_numeric", pd.Series(dtype=float))).dropna().to_numpy(dtype=float)
            if len(metric_values) == 0:
                continue

            if direction == "lower":
                top1 = float(np.min(metric_values))
                cutoff = float(np.max(metric_values))
                estimated_rank = 1 + int(np.sum(metric_values < best_value))
                gap_top1 = best_value - top1
                gap_top10 = best_value - cutoff
            else:
                top1 = float(np.max(metric_values))
                cutoff = float(np.min(metric_values))
                estimated_rank = 1 + int(np.sum(metric_values > best_value))
                gap_top1 = top1 - best_value
                gap_top10 = cutoff - best_value

            rows.append(
                {
                    "dataset": dataset_name,
                    "benchmark_id": benchmark_id,
                    "leaderboard_metric_name": metric_name,
                    "metric_kind": metric_kind,
                    "direction": direction,
                    "best_model": str(best_row.get("model", "")),
                    "best_workflow": str(best_row.get("workflow", "")),
                    "best_value": best_value,
                    "best_value_raw_model_metric": best_value_raw,
                    "best_value_metric_column": metric_col,
                    "leaderboard_top1": top1,
                    "leaderboard_top10_cutoff": cutoff,
                    "estimated_rank_vs_top10": int(estimated_rank) if estimated_rank <= 10 else 11,
                    "in_top10_estimate": bool(estimated_rank <= 10),
                    "gap_to_top1": float(gap_top1),
                    "gap_to_top10_cutoff": float(gap_top10),
                }
            )

    leaderboard_eval = pd.DataFrame(rows)

if leaderboard_eval is None or leaderboard_eval.empty:
    display_note("No leaderboard comparison was available for this run.")
else:
    leaderboard_eval = leaderboard_eval.copy()
    if "estimated_rank_vs_top10" in leaderboard_eval.columns:
        leaderboard_eval["estimated_rank_vs_top10"] = pd.to_numeric(leaderboard_eval["estimated_rank_vs_top10"], errors="coerce")
        leaderboard_eval["estimated_rank_label"] = leaderboard_eval["estimated_rank_vs_top10"].apply(lambda x: ">10" if pd.notna(x) and int(x) > 10 else (str(int(x)) if pd.notna(x) else "NA"))
        if "in_top10_estimate" not in leaderboard_eval.columns:
            leaderboard_eval["in_top10_estimate"] = leaderboard_eval["estimated_rank_vs_top10"].fillna(11).astype(float) <= 10

    missing_rank_mask = pd.Series(False, index=leaderboard_eval.index)
    if "estimated_rank_vs_top10" in leaderboard_eval.columns:
        missing_rank_mask = missing_rank_mask | pd.to_numeric(leaderboard_eval["estimated_rank_vs_top10"], errors="coerce").isna()
    if "gap_to_top10_cutoff" in leaderboard_eval.columns:
        missing_rank_mask = missing_rank_mask | pd.to_numeric(leaderboard_eval["gap_to_top10_cutoff"], errors="coerce").isna()

    if bool(missing_rank_mask.any()) and "dataset" in SUMMARY_METRICS.columns:
        detail = SUMMARY_METRICS.copy()
        if "primary_metric_value" in detail.columns:
            detail["primary_metric_value"] = pd.to_numeric(detail["primary_metric_value"], errors="coerce")
            detail = detail.loc[detail["primary_metric_value"].notna()].copy()

        for row_idx in leaderboard_eval.index[missing_rank_mask]:
            dataset_name = str(leaderboard_eval.at[row_idx, "dataset"]) if "dataset" in leaderboard_eval.columns else ""
            if not dataset_name:
                continue

            ds_rows = detail.loc[detail["dataset"].astype(str) == dataset_name].copy()
            if ds_rows.empty:
                continue

            rank_col = "leaderboard_estimated_rank_vs_top10" if "leaderboard_estimated_rank_vs_top10" in ds_rows.columns else None
            gap_col = "leaderboard_gap_to_top10_cutoff" if "leaderboard_gap_to_top10_cutoff" in ds_rows.columns else None
            if rank_col is not None:
                ds_rows[rank_col] = pd.to_numeric(ds_rows[rank_col], errors="coerce")
            if gap_col is not None:
                ds_rows[gap_col] = pd.to_numeric(ds_rows[gap_col], errors="coerce")

            comparable_mask = pd.Series(False, index=ds_rows.index)
            if rank_col is not None:
                comparable_mask = comparable_mask | ds_rows[rank_col].notna()
            if gap_col is not None:
                comparable_mask = comparable_mask | ds_rows[gap_col].notna()
            ds_rows = ds_rows.loc[comparable_mask].copy()
            if ds_rows.empty:
                continue

            target_metric = ""
            if "leaderboard_metric_name" in ds_rows.columns:
                lb_metric_series = ds_rows["leaderboard_metric_name"].astype(str).apply(leaderboard_metric_kind)
                lb_metric_series = lb_metric_series[lb_metric_series.notna() & (lb_metric_series.astype(str) != "")]
                if not lb_metric_series.empty:
                    target_metric = str(lb_metric_series.iloc[0]).strip().lower()

            if not target_metric and "primary_metric" in ds_rows.columns:
                primary_metric_series = ds_rows["primary_metric"].astype(str).apply(leaderboard_metric_kind)
                primary_metric_series = primary_metric_series[primary_metric_series.notna() & (primary_metric_series.astype(str) != "")]
                if not primary_metric_series.empty:
                    target_metric = str(primary_metric_series.iloc[0]).strip().lower()

            if target_metric and "primary_metric" in ds_rows.columns:
                metric_mask = ds_rows["primary_metric"].astype(str).apply(leaderboard_metric_kind) == target_metric
                if bool(metric_mask.any()):
                    ds_rows = ds_rows.loc[metric_mask].copy()

            direction = leaderboard_direction(target_metric)
            ascending = False if direction == "higher" else True
            best_row = ds_rows.sort_values("primary_metric_value", ascending=ascending).iloc[0]

            if "best_model" in leaderboard_eval.columns:
                leaderboard_eval.at[row_idx, "best_model"] = best_row.get("model", leaderboard_eval.at[row_idx, "best_model"])
            if "best_workflow" in leaderboard_eval.columns:
                leaderboard_eval.at[row_idx, "best_workflow"] = best_row.get("workflow", leaderboard_eval.at[row_idx, "best_workflow"])
            if "primary_metric" in leaderboard_eval.columns and target_metric:
                leaderboard_eval.at[row_idx, "primary_metric"] = target_metric
            if "best_value_raw_model_metric" in leaderboard_eval.columns:
                leaderboard_eval.at[row_idx, "best_value_raw_model_metric"] = pd.to_numeric(best_row.get("primary_metric_value", np.nan), errors="coerce")
            if "best_value" in leaderboard_eval.columns:
                leaderboard_eval.at[row_idx, "best_value"] = pd.to_numeric(best_row.get("primary_metric_value", np.nan), errors="coerce")
            if "best_value_metric_column" in leaderboard_eval.columns and "primary_metric" in best_row.index:
                leaderboard_eval.at[row_idx, "best_value_metric_column"] = str(best_row.get("primary_metric", ""))
            if "leaderboard_metric_name" in leaderboard_eval.columns and "leaderboard_metric_name" in best_row.index:
                leaderboard_eval.at[row_idx, "leaderboard_metric_name"] = str(best_row.get("leaderboard_metric_name", ""))
            if "leaderboard_top1" in leaderboard_eval.columns and "leaderboard_metric_reference" in best_row.index:
                leaderboard_eval.at[row_idx, "leaderboard_top1"] = pd.to_numeric(best_row.get("leaderboard_metric_reference", np.nan), errors="coerce")
            if "gap_to_top1" in leaderboard_eval.columns and "leaderboard_delta_primary" in best_row.index:
                leaderboard_eval.at[row_idx, "gap_to_top1"] = pd.to_numeric(best_row.get("leaderboard_delta_primary", np.nan), errors="coerce")

            if "estimated_rank_vs_top10" in leaderboard_eval.columns and rank_col is not None:
                leaderboard_eval.at[row_idx, "estimated_rank_vs_top10"] = pd.to_numeric(best_row.get(rank_col, np.nan), errors="coerce")
            if "in_top10_estimate" in leaderboard_eval.columns:
                in_top10_val = best_row.get("leaderboard_estimated_in_top10", np.nan)
                if pd.notna(in_top10_val):
                    leaderboard_eval.at[row_idx, "in_top10_estimate"] = bool(in_top10_val)
            if "gap_to_top10_cutoff" in leaderboard_eval.columns and gap_col is not None:
                leaderboard_eval.at[row_idx, "gap_to_top10_cutoff"] = pd.to_numeric(best_row.get(gap_col, np.nan), errors="coerce")

            if "leaderboard_top10_cutoff" in leaderboard_eval.columns and "best_value" in leaderboard_eval.columns and "gap_to_top10_cutoff" in leaderboard_eval.columns:
                best_value = pd.to_numeric(leaderboard_eval.at[row_idx, "best_value"], errors="coerce")
                gap_value = pd.to_numeric(leaderboard_eval.at[row_idx, "gap_to_top10_cutoff"], errors="coerce")
                cutoff_value = pd.to_numeric(leaderboard_eval.at[row_idx, "leaderboard_top10_cutoff"], errors="coerce")
                if pd.isna(cutoff_value) and pd.notna(best_value) and pd.notna(gap_value):
                    if direction == "higher":
                        leaderboard_eval.at[row_idx, "leaderboard_top10_cutoff"] = float(best_value + gap_value)
                    else:
                        leaderboard_eval.at[row_idx, "leaderboard_top10_cutoff"] = float(best_value - gap_value)

    if "estimated_rank_vs_top10" in leaderboard_eval.columns:
        leaderboard_eval["estimated_rank_vs_top10"] = pd.to_numeric(leaderboard_eval["estimated_rank_vs_top10"], errors="coerce")
        leaderboard_eval["estimated_rank_label"] = leaderboard_eval["estimated_rank_vs_top10"].apply(lambda x: ">10" if pd.notna(x) and int(x) > 10 else (str(int(x)) if pd.notna(x) else "NA"))
    if "in_top10_estimate" not in leaderboard_eval.columns and "estimated_rank_vs_top10" in leaderboard_eval.columns:
        leaderboard_eval["in_top10_estimate"] = leaderboard_eval["estimated_rank_vs_top10"].fillna(11).astype(float) <= 10

    total = int(len(leaderboard_eval))
    rank_series = pd.to_numeric(leaderboard_eval.get("estimated_rank_vs_top10", pd.Series(dtype=float)), errors="coerce")
    comparable_mask = rank_series.notna()
    if not comparable_mask.any() and "gap_to_top10_cutoff" in leaderboard_eval.columns:
        comparable_mask = pd.to_numeric(leaderboard_eval["gap_to_top10_cutoff"], errors="coerce").notna()
    comparable = int(comparable_mask.sum())

    in_top10_series = pd.to_numeric(leaderboard_eval.get("in_top10_estimate", pd.Series(dtype=float)), errors="coerce")
    in_top10 = int(in_top10_series[comparable_mask].fillna(0).astype(int).sum()) if comparable > 0 else 0
    top1_count = int((rank_series[comparable_mask] == 1).sum()) if comparable > 0 else 0

    display_note(
        f"### Leaderboard Performance Overview\n"
        f"Datasets in this run: **{total}**  \n"
        f"Datasets with leaderboard-comparable metrics: **{comparable}**  \n"
        f"Estimated in top-10: **{in_top10}/{comparable if comparable > 0 else 1}**  \n"
        f"Estimated #1: **{top1_count}**"
    )

    display_cols = [
        "dataset",
        "leaderboard_metric_name",
        "best_model",
        "best_workflow",
        "best_value",
        "best_value_raw_model_metric",
        "best_value_metric_column",
        "leaderboard_top1",
        "leaderboard_top10_cutoff",
        "estimated_rank_label",
        "in_top10_estimate",
        "gap_to_top1",
        "gap_to_top10_cutoff",
    ]
    display_cols = [c for c in display_cols if c in leaderboard_eval.columns]
    sort_cols = []
    sort_ascending = []
    if "in_top10_estimate" in leaderboard_eval.columns:
        sort_cols.append("in_top10_estimate")
        sort_ascending.append(False)
    if "gap_to_top10_cutoff" in leaderboard_eval.columns:
        sort_cols.append("gap_to_top10_cutoff")
        sort_ascending.append(True)

    display_df = leaderboard_eval[display_cols].copy()
    if sort_cols:
        display_df = display_df.sort_values(by=sort_cols, ascending=sort_ascending)
    display(display_df.reset_index(drop=True))

    if "gap_to_top10_cutoff" in leaderboard_eval.columns:
        plot_df = leaderboard_eval.sort_values("gap_to_top10_cutoff", ascending=True).copy()
        fig = px.bar(
            plot_df,
            x="dataset",
            y="gap_to_top10_cutoff",
            color="metric_kind" if "metric_kind" in plot_df.columns else None,
            hover_data=[col for col in ["best_model", "best_value", "leaderboard_top10_cutoff", "estimated_rank_label"] if col in plot_df.columns],
            title="Gap to leaderboard top-10 cutoff (negative = better than cutoff)",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(yaxis_title="Gap to top-10 cutoff", xaxis_title="Dataset")
        fig.show()

    if "estimated_rank_label" in leaderboard_eval.columns:
        rank_order = [str(i) for i in range(1, 11)] + [">10"]
        rank_counts = (
            leaderboard_eval.groupby("estimated_rank_label", dropna=False)
            .size()
            .reset_index(name="datasets")
        )
        rank_counts["estimated_rank_label"] = pd.Categorical(rank_counts["estimated_rank_label"], categories=rank_order, ordered=True)
        rank_counts = rank_counts.sort_values("estimated_rank_label")
        fig = px.bar(rank_counts, x="estimated_rank_label", y="datasets", title="Estimated leaderboard rank distribution")
        fig.update_layout(xaxis_title="Estimated rank vs top-10", yaxis_title="Dataset count")
        fig.show()


No leaderboard comparison was available for this run.

In [74]:
# Run completeness and leaderboard-reference context

if RUN_STATUS is None or RUN_STATUS.empty:
    display_note("No per-dataset `run_status.json` files were found in this run directory.")
else:
    status_df = RUN_STATUS.copy()
    status_df["status"] = status_df["status"].astype(str)
    display_note("### Run completeness by dataset")
    display(status_df.sort_values(["status", "dataset"], ascending=[True, True]).reset_index(drop=True))

    status_counts = status_df.groupby("status", dropna=False).size().reset_index(name="datasets")
    fig = px.bar(status_counts, x="status", y="datasets", title="Run status counts")
    fig.update_layout(xaxis_title="Status", yaxis_title="Dataset count")
    fig.show()

if LEADERBOARD_TOP10 is None or LEADERBOARD_TOP10.empty:
    display_note("No `leaderboard_top10_reference.csv` was found for this run.")
else:
    ref = LEADERBOARD_TOP10.copy()
    if "is_top10_entry" in ref.columns:
        ref = ref.loc[ref["is_top10_entry"].fillna(False)].copy()

    display_note("### Leaderboard reference coverage")
    coverage_cols = [
        "dataset",
        "benchmark_id",
        "benchmark_suite",
        "leaderboard_metric_name",
        "leaderboard_dataset_split",
        "rank",
        "model",
        "metric_value",
    ]
    coverage_cols = [c for c in coverage_cols if c in ref.columns]
    display(ref[coverage_cols].head(30))

    coverage_summary = (
        ref.groupby([c for c in ["dataset", "leaderboard_metric_name"] if c in ref.columns], dropna=False)
        .agg(
            top10_rows=("model", "size"),
            top10_best=("metric_value_numeric", "min"),
            top10_worst=("metric_value_numeric", "max"),
        )
        .reset_index()
    )
    display(coverage_summary.sort_values("dataset").reset_index(drop=True))


### Run completeness by dataset

,dataset,status,checkpoint_stage,n_metrics_rows,started_at,completed_at,elapsed_seconds
0,tdc_carcinogens_lagunin,running,deep:ChemML MLP (PyTorch),12,2026-04-24 14:48:01,,NaN


### Leaderboard reference coverage

,dataset,benchmark_id,benchmark_suite,leaderboard_metric_name,leaderboard_dataset_split,rank,model,metric_value
0,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,1,CFA,0.576 ± 0.025
1,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,2,MapLight,0.562 ± 0.008
2,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,3,MapLight + GNN,0.557 ± 0.034
3,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,4,Euclia ML model,0.547 ± 0.032
4,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,5,"Voting Regressor (KNN, SVM)",0.544 ± 0.034
5,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,6,MiniMol,0.495 ± 0.042
6,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,7,DeepMol (AutoML),0.485 ± 0.039
7,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,8,Basic ML,0.438 ± 0.011
8,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,9,SimGCN,0.392 ± 0.065
9,tdc_half_life_obach,half_life_obach,tdc,Spearman,Scaffold,10,ADMETrix,0.372 ± 0.026


,dataset,leaderboard_metric_name,top10_rows,top10_best,top10_worst
0,esol_delaney,Test RMSE,2,0.8851,1.7406
1,lipophilicity,Test RMSE,2,0.7806,0.9621
2,polaris_adme_fang_hppb_1,mean_squared_error,10,0.1430,0.3830
3,polaris_adme_fang_perm_1,mean_squared_error,10,0.1130,0.2570
4,polaris_adme_fang_rclint_1,mean_squared_error,10,0.2160,0.4030
5,polaris_adme_fang_rppb_1,mean_squared_error,10,0.2300,0.6340
6,polaris_adme_fang_solu_1,mean_squared_error,10,0.2220,0.3290
7,tdc_caco2_wang,MAE,10,0.2560,0.3260
8,tdc_clearance_hepatocyte_az,Spearman,10,0.4300,0.5360
9,tdc_clearance_microsome_az,Spearman,10,0.5780,0.6300


### Cost vs value diagnostics across runs

This section compares all run folders under `benchmark_results/` and summarizes runtime cost vs performance value.

- Run-level value prefers leaderboard-relative metrics when available (`leaderboard_delta_top10_best`, then `leaderboard_delta_primary`) and falls back to held-out RMSE.
- Feature-selection scaling uses explicit selector-stage timings when available and otherwise uses a clearly labeled proxy (`selector timeout cap` or `time-to-first-model`).
        


In [75]:
# COST_VS_VALUE_DIAGNOSTICS_MARKER
# Cross-run cost/value diagnostics + feature-selection scaling diagnostics.

benchmark_root = RUN_DIR.parent if RUN_DIR.parent.name == "benchmark_results" else Path("./benchmark_results").resolve()
run_dirs = sorted([p for p in benchmark_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime)


def load_run_metrics(run_dir):
    summary = standardize_metric_frame(read_optional_csv(run_dir / "summary_metrics.csv"))
    if summary is not None and not summary.empty:
        return summary

    parts = []
    for metrics_path in sorted(run_dir.glob("*/metrics.csv")):
        dataset_name = metrics_path.parent.name
        frame = standardize_metric_frame(pd.read_csv(metrics_path))
        if frame is None or frame.empty:
            continue
        if "dataset" not in frame.columns:
            frame["dataset"] = dataset_name
        parts.append(frame)

    if not parts:
        return None
    return pd.concat(parts, ignore_index=True)


def load_run_status(run_dir):
    rows = []
    for status_path in sorted(run_dir.glob("*/run_status.json")):
        payload = read_optional_json(status_path)
        if not isinstance(payload, dict):
            continue
        rows.append(
            {
                "dataset": status_path.parent.name,
                "status": payload.get("status", "unknown"),
                "checkpoint_stage": payload.get("checkpoint_stage", ""),
                "started_at": payload.get("started_at", ""),
                "completed_at": payload.get("completed_at", ""),
                "elapsed_seconds": payload.get("elapsed_seconds", np.nan),
            }
        )
    status_df = pd.DataFrame(rows)
    if not status_df.empty and "elapsed_seconds" in status_df.columns:
        status_df["elapsed_seconds"] = pd.to_numeric(status_df["elapsed_seconds"], errors="coerce")
    return status_df


def bool_like_series(series):
    if series is None:
        return pd.Series(dtype=bool)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes", "y", "t"})


def parse_stage_runtime_seconds(stage_frame):
    if stage_frame is None or stage_frame.empty:
        return np.nan
    label_col = pick_column(stage_frame.columns, ["stage_label", "stage", "step", "name"])
    seconds_col = pick_column(stage_frame.columns, ["elapsed_seconds", "duration_seconds", "seconds", "elapsed"])
    if label_col is None or seconds_col is None:
        return np.nan

    labels = stage_frame[label_col].astype(str).str.lower()
    mask = labels.str.contains("feature") | labels.str.contains("select")
    selector_rows = stage_frame.loc[mask].copy()
    if selector_rows.empty:
        return np.nan

    selector_rows[seconds_col] = pd.to_numeric(selector_rows[seconds_col], errors="coerce")
    return float(selector_rows[seconds_col].sum()) if selector_rows[seconds_col].notna().any() else np.nan


run_rows = []
dataset_cost_value_rows = []
selector_rows = []

for run_dir in run_dirs:
    run_name = run_dir.name
    run_metrics = load_run_metrics(run_dir)
    if run_metrics is None or run_metrics.empty:
        continue

    run_metrics = run_metrics.copy()
    rmse_col_run = metric_column(run_metrics, "AUTO", metric_hint="rmse")
    if rmse_col_run is None:
        continue

    run_metrics[rmse_col_run] = pd.to_numeric(run_metrics[rmse_col_run], errors="coerce")

    numeric_candidates = [
        "elapsed_seconds",
        "n_molecules",
        "n_train",
        "n_test",
        "selected_feature_count",
        "original_feature_count",
        "leaderboard_delta_top10_best",
        "leaderboard_delta_primary",
        "leaderboard_rank",
        "leaderboard_estimated_rank_vs_top10",
        "leaderboard_gap_to_top10_cutoff",
        "selector_elasticnet_timeout_seconds",
    ]
    for col in numeric_candidates:
        if col in run_metrics.columns:
            run_metrics[col] = pd.to_numeric(run_metrics[col], errors="coerce")

    run_status = load_run_status(run_dir)

    best_by_dataset = (
        run_metrics.sort_values(rmse_col_run, ascending=True, na_position="last")
        .groupby("dataset", as_index=False)
        .first()
    )

    dataset_elapsed_map = {}
    if not run_status.empty and "dataset" in run_status.columns and "elapsed_seconds" in run_status.columns:
        dataset_elapsed_map = run_status.set_index("dataset")["elapsed_seconds"].to_dict()

    status_completed_count = np.nan
    if not run_status.empty and "status" in run_status.columns:
        status_completed_count = int((run_status["status"].astype(str).str.lower() == "completed").sum())

    total_elapsed_seconds = np.nan
    if not run_status.empty and "elapsed_seconds" in run_status.columns and run_status["elapsed_seconds"].notna().any():
        total_elapsed_seconds = float(run_status["elapsed_seconds"].sum())

    if not np.isfinite(total_elapsed_seconds):
        if "dataset" in run_metrics.columns and "elapsed_seconds" in run_metrics.columns:
            elapsed_proxy = run_metrics.groupby("dataset", dropna=False)["elapsed_seconds"].max()
            if elapsed_proxy.notna().any():
                total_elapsed_seconds = float(elapsed_proxy.sum())

    delta_col = pick_column(best_by_dataset.columns, ["leaderboard_delta_top10_best", "leaderboard_delta_primary"])
    rank_col = pick_column(best_by_dataset.columns, ["leaderboard_estimated_rank_vs_top10", "leaderboard_rank"])
    top10_col = pick_column(best_by_dataset.columns, ["leaderboard_estimated_in_top10", "leaderboard_in_top10"])

    delta_values = pd.to_numeric(best_by_dataset[delta_col], errors="coerce") if delta_col is not None else pd.Series(dtype=float)
    rank_values = pd.to_numeric(best_by_dataset[rank_col], errors="coerce") if rank_col is not None else pd.Series(dtype=float)

    top10_rate = np.nan
    if top10_col is not None:
        bool_series = best_by_dataset[top10_col].astype(str).str.lower().map({"true": True, "false": False})
        if bool_series.notna().any():
            top10_rate = float(bool_series.mean())

    run_rows.append(
        {
            "run_name": run_name,
            "run_mtime": pd.Timestamp(run_dir.stat().st_mtime, unit="s"),
            "datasets": int(best_by_dataset["dataset"].nunique()),
            "completed_datasets": status_completed_count,
            "models": int(run_metrics["model"].nunique()) if "model" in run_metrics.columns else np.nan,
            "total_elapsed_seconds": total_elapsed_seconds,
            "total_elapsed_hours": total_elapsed_seconds / 3600.0 if np.isfinite(total_elapsed_seconds) else np.nan,
            "mean_best_rmse": float(best_by_dataset[rmse_col_run].mean(skipna=True)),
            "median_best_rmse": float(best_by_dataset[rmse_col_run].median(skipna=True)),
            "mean_leaderboard_delta": float(delta_values.mean(skipna=True)) if len(delta_values) and delta_values.notna().any() else np.nan,
            "median_leaderboard_delta": float(delta_values.median(skipna=True)) if len(delta_values) and delta_values.notna().any() else np.nan,
            "mean_estimated_rank": float(rank_values.mean(skipna=True)) if len(rank_values) and rank_values.notna().any() else np.nan,
            "top10_hit_rate": top10_rate,
            "value_metric_used": delta_col if delta_col is not None else rmse_col_run,
        }
    )

    # Explicit selector stage timings (if emitted by future benchmark runs)
    explicit_selector_seconds_by_dataset = {}
    step_runtime_summary = read_optional_csv(run_dir / "step_runtime_summary.csv")
    if step_runtime_summary is not None and not step_runtime_summary.empty:
        ds_col = pick_column(step_runtime_summary.columns, ["dataset", "dataset_name"])
        label_col = pick_column(step_runtime_summary.columns, ["stage_label", "stage", "step", "name"])
        sec_col = pick_column(step_runtime_summary.columns, ["elapsed_seconds", "duration_seconds", "seconds", "elapsed"])
        if ds_col is not None and label_col is not None and sec_col is not None:
            tmp = step_runtime_summary.copy()
            tmp[sec_col] = pd.to_numeric(tmp[sec_col], errors="coerce")
            tmp = tmp.loc[tmp[label_col].astype(str).str.lower().str.contains("feature|select", regex=True)].copy()
            if not tmp.empty:
                grouped = tmp.groupby(ds_col, dropna=False)[sec_col].sum(min_count=1)
                explicit_selector_seconds_by_dataset.update(grouped.to_dict())

    for step_path in run_dir.glob("*/step_runtime.csv"):
        dataset_name = step_path.parent.name
        if dataset_name in explicit_selector_seconds_by_dataset:
            continue
        stage_frame = read_optional_csv(step_path)
        seconds = parse_stage_runtime_seconds(stage_frame)
        if np.isfinite(seconds):
            explicit_selector_seconds_by_dataset[dataset_name] = float(seconds)

    for dataset_name, dataset_frame in run_metrics.groupby("dataset", dropna=False):
        dataset_frame = dataset_frame.copy()
        dataset_frame = dataset_frame.sort_values(rmse_col_run, ascending=True, na_position="last")
        best_row = dataset_frame.iloc[0]

        dataset_elapsed = dataset_elapsed_map.get(dataset_name, np.nan)
        if not np.isfinite(dataset_elapsed) and "elapsed_seconds" in dataset_frame.columns:
            dataset_elapsed = float(dataset_frame["elapsed_seconds"].max(skipna=True))

        selector_seconds = np.nan
        selector_seconds_source = "unavailable"

        if dataset_name in explicit_selector_seconds_by_dataset:
            selector_seconds = float(explicit_selector_seconds_by_dataset[dataset_name])
            selector_seconds_source = "stage_runtime"
        else:
            timed_out = False
            if "selector_timed_out" in dataset_frame.columns:
                timed_out = bool(bool_like_series(dataset_frame["selector_timed_out"]).any())

            timeout_cap = np.nan
            if "selector_elasticnet_timeout_seconds" in dataset_frame.columns:
                timeout_candidates = pd.to_numeric(dataset_frame["selector_elasticnet_timeout_seconds"], errors="coerce")
                if timeout_candidates.notna().any():
                    timeout_cap = float(timeout_candidates.dropna().iloc[0])

            if timed_out and np.isfinite(timeout_cap):
                selector_seconds = timeout_cap
                selector_seconds_source = "selector_timeout_cap"
            elif "elapsed_seconds" in dataset_frame.columns:
                first_elapsed = float(pd.to_numeric(dataset_frame["elapsed_seconds"], errors="coerce").min(skipna=True))
                if np.isfinite(first_elapsed):
                    selector_seconds = first_elapsed
                    selector_seconds_source = "first_model_elapsed_proxy"

        n_molecules_value = best_row.get("n_molecules", np.nan)
        n_train_value = best_row.get("n_train", np.nan)
        n_test_value = best_row.get("n_test", np.nan)

        selector_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "n_molecules": pd.to_numeric(n_molecules_value, errors="coerce"),
                "n_train": pd.to_numeric(n_train_value, errors="coerce"),
                "n_test": pd.to_numeric(n_test_value, errors="coerce"),
                "selector_seconds": pd.to_numeric(selector_seconds, errors="coerce"),
                "selector_seconds_source": selector_seconds_source,
                "selector_timed_out": bool(bool_like_series(dataset_frame.get("selector_timed_out", pd.Series([False]))).any()),
                "selector_method": best_row.get("selector_method", ""),
                "selected_feature_count": pd.to_numeric(best_row.get("selected_feature_count", np.nan), errors="coerce"),
                "original_feature_count": pd.to_numeric(best_row.get("original_feature_count", np.nan), errors="coerce"),
            }
        )

        dataset_cost_value_rows.append(
            {
                "run_name": run_name,
                "dataset": dataset_name,
                "dataset_elapsed_seconds": pd.to_numeric(dataset_elapsed, errors="coerce"),
                "best_rmse": pd.to_numeric(best_row.get(rmse_col_run, np.nan), errors="coerce"),
                "leaderboard_delta_top10_best": pd.to_numeric(best_row.get("leaderboard_delta_top10_best", np.nan), errors="coerce"),
                "leaderboard_delta_primary": pd.to_numeric(best_row.get("leaderboard_delta_primary", np.nan), errors="coerce"),
                "leaderboard_estimated_rank_vs_top10": pd.to_numeric(best_row.get("leaderboard_estimated_rank_vs_top10", np.nan), errors="coerce"),
                "n_molecules": pd.to_numeric(best_row.get("n_molecules", np.nan), errors="coerce"),
            }
        )

RUN_COST_VALUE = pd.DataFrame(run_rows)
DATASET_COST_VALUE = pd.DataFrame(dataset_cost_value_rows)
SELECTOR_SCALING = pd.DataFrame(selector_rows)

if RUN_COST_VALUE.empty:
    display_note("No benchmark runs were detected under `benchmark_results/` with usable metrics.")
else:
    run_table = RUN_COST_VALUE.copy().sort_values("run_mtime", ascending=False)

    if not SELECTOR_SCALING.empty:
        selector_agg = (
            SELECTOR_SCALING.groupby("run_name", dropna=False)
            .agg(
                selector_timeout_rate=("selector_timed_out", "mean"),
                mean_selector_seconds=("selector_seconds", "mean"),
                median_selector_seconds=("selector_seconds", "median"),
                selector_rows=("dataset", "size"),
            )
            .reset_index()
        )
        run_table = run_table.merge(selector_agg, on="run_name", how="left")

    display_note("### Run-level cost vs value")
    display(
        run_table[
            [
                col
                for col in [
                    "run_name",
                    "run_mtime",
                    "datasets",
                    "completed_datasets",
                    "models",
                    "total_elapsed_hours",
                    "mean_best_rmse",
                    "mean_leaderboard_delta",
                    "mean_estimated_rank",
                    "top10_hit_rate",
                    "selector_timeout_rate",
                    "mean_selector_seconds",
                    "value_metric_used",
                ]
                if col in run_table.columns
            ]
        ].reset_index(drop=True)
    )

    run_plot_df = run_table.copy()
    if run_plot_df["total_elapsed_hours"].notna().sum() >= 2:
        value_col = "mean_leaderboard_delta" if run_plot_df["mean_leaderboard_delta"].notna().sum() >= 2 else "mean_best_rmse"
        fig = px.scatter(
            run_plot_df,
            x="total_elapsed_hours",
            y=value_col,
            color="run_name",
            hover_data=[c for c in ["datasets", "completed_datasets", "models", "mean_selector_seconds", "selector_timeout_rate"] if c in run_plot_df.columns],
            title=f"Run-level cost vs value ({value_col})",
        )
        fig.update_layout(xaxis_title="Total runtime across datasets (hours)", yaxis_title=value_col)
        fig.show()

    if not DATASET_COST_VALUE.empty:
        dataset_plot = DATASET_COST_VALUE.copy()
        dataset_plot["dataset_elapsed_hours"] = dataset_plot["dataset_elapsed_seconds"] / 3600.0
        value_col_dataset = (
            "leaderboard_delta_top10_best"
            if dataset_plot["leaderboard_delta_top10_best"].notna().sum() >= 3
            else "best_rmse"
        )
        fig = px.scatter(
            dataset_plot,
            x="dataset_elapsed_hours",
            y=value_col_dataset,
            color="run_name",
            hover_data=[c for c in ["dataset", "n_molecules", "leaderboard_estimated_rank_vs_top10"] if c in dataset_plot.columns],
            title=f"Dataset-level runtime vs value ({value_col_dataset})",
        )
        fig.update_layout(xaxis_title="Dataset runtime (hours)", yaxis_title=value_col_dataset)
        fig.show()

if SELECTOR_SCALING.empty:
    display_note("No selector diagnostics could be inferred for these runs.")
else:
    selector_df = SELECTOR_SCALING.copy()
    selector_df["n_molecules"] = pd.to_numeric(selector_df["n_molecules"], errors="coerce")
    selector_df["selector_seconds"] = pd.to_numeric(selector_df["selector_seconds"], errors="coerce")
    selector_df["selector_hours"] = selector_df["selector_seconds"] / 3600.0

    display_note("### Feature-selection time vs dataset size")
    display(
        selector_df.sort_values("selector_seconds", ascending=False)
        [[
            col
            for col in [
                "run_name",
                "dataset",
                "n_molecules",
                "selector_seconds",
                "selector_hours",
                "selector_seconds_source",
                "selector_timed_out",
                "selector_method",
                "selected_feature_count",
                "original_feature_count",
            ]
            if col in selector_df.columns
        ]]
        .reset_index(drop=True)
    )

    fit_df = selector_df.loc[
        selector_df["n_molecules"].gt(0)
        & selector_df["selector_seconds"].gt(0)
    ].copy()

    if len(fit_df) >= 3:
        log_x = np.log10(fit_df["n_molecules"].to_numpy(dtype=float))
        log_y = np.log10(fit_df["selector_seconds"].to_numpy(dtype=float))
        corr = float(np.corrcoef(log_x, log_y)[0, 1])
        slope, intercept = np.polyfit(log_x, log_y, 1)
        print(
            "Selector scaling fit (log10 seconds vs log10 n_molecules): "
            f"slope={slope:.3f}, intercept={intercept:.3f}, corr={corr:.3f}"
        )

        fig = px.scatter(
            fit_df,
            x="n_molecules",
            y="selector_seconds",
            color="run_name",
            symbol="selector_seconds_source",
            hover_data=[c for c in ["dataset", "selected_feature_count", "selector_timed_out"] if c in fit_df.columns],
            title="Feature-selection cost scaling across runs",
        )
        fig.update_xaxes(type="log", title="Dataset size (n_molecules, log scale)")
        fig.update_yaxes(type="log", title="Selector runtime (seconds, log scale)")
        fig.show()
    else:
        display_note("Insufficient positive selector-time samples to fit a size-scaling trend.")

    source_counts = (
        selector_df.groupby("selector_seconds_source", dropna=False)
        .size()
        .reset_index(name="datasets")
        .sort_values("datasets", ascending=False)
    )
    display_note("Selector-time source diagnostics")
    display(source_counts)

    if "first_model_elapsed_proxy" in source_counts["selector_seconds_source"].astype(str).tolist():
        display_note(
            "`first_model_elapsed_proxy` indicates an estimated upper bound for selector time when explicit stage timings were not saved "
            "for that run."
        )
        


### Run-level cost vs value

,run_name,run_mtime,datasets,completed_datasets,models,total_elapsed_hours,mean_best_rmse,mean_leaderboard_delta,mean_estimated_rank,top10_hit_rate,selector_timeout_rate,mean_selector_seconds,value_metric_used
0,all_benchmarks_no_unimol_20260425_223505,2026-04-24 21:51:31.946180820,1,0,14,0.121124,0.332553,NaN,NaN,NaN,0.000000,173.879000,leaderboard_delta_top10_best
1,all_benchmarks_no_unimol_20260424_223505,2026-04-24 21:24:58.125092030,18,17,21,4.105885,3.334543,-0.090495,5.400000,0.800,0.000000,174.001444,leaderboard_delta_top10_best
2,all_benchmarks_no_unimol_20260423_223505,2026-04-24 14:07:52.368180990,19,18,19,8.230427,6.374634,-0.045230,5.875000,0.750,0.000000,780.963632,leaderboard_delta_top10_best
3,all_benchmarks_no_unimol_20260422_223505,2026-04-23 15:43:19.848572493,19,17,17,6.052271,6.331432,0.015127,5.000000,0.875,0.052632,779.231105,leaderboard_delta_top10_best
4,all_benchmarks_no_unimol_20260421_223505,2026-04-22 13:14:19.702852488,15,14,15,8.292190,7.947044,-0.007785,5.583333,0.750,0.000000,1039.974733,leaderboard_delta_top10_best
5,all_benchmarks_no_unimol_20260419_223505,2026-04-22 03:04:24.155577660,19,19,15,31.070296,6.540224,0.055222,6.937500,0.625,0.210526,4785.402368,leaderboard_delta_top10_best
6,all_benchmarks_full_no_unimol_20260418_095237,2026-04-20 05:33:35.828924179,12,12,14,17.207416,7.931306,-0.209005,5.750000,0.625,0.000000,4491.175583,leaderboard_delta_top10_best


### Feature-selection time vs dataset size

,run_name,dataset,n_molecules,selector_seconds,selector_hours,selector_seconds_source,selector_timed_out,selector_method,selected_feature_count,original_feature_count
0,all_benchmarks_full_no_unimol_20260418_095237,tdc_lipophilicity_astrazeneca,4200,18728.099,5.202250,first_model_elapsed_proxy,False,elasticnet_cv,420,19331
1,all_benchmarks_no_unimol_20260419_223505,tdc_solubility_aqsoldb,9980,16628.423,4.619006,first_model_elapsed_proxy,True,random_forest_importance_fallback,998,19331
2,all_benchmarks_no_unimol_20260419_223505,tdc_ld50_zhu,7385,13994.970,3.887492,first_model_elapsed_proxy,True,random_forest_importance_fallback,739,19331
3,all_benchmarks_full_no_unimol_20260418_095237,tdc_ld50_zhu,7385,13987.766,3.885491,first_model_elapsed_proxy,False,random_forest_importance_fallback,512,10115
4,all_benchmarks_no_unimol_20260419_223505,tdc_lipophilicity_astrazeneca,4200,13972.269,3.881186,first_model_elapsed_proxy,False,random_forest_importance_fallback,420,19331
...,...,...,...,...,...,...,...,...,...,...
98,all_benchmarks_no_unimol_20260422_223505,freesolv_sampl,642,34.925,0.009701,stage_runtime,False,elasticnet_cv,451,10115
99,all_benchmarks_no_unimol_20260422_223505,polaris_adme_fang_solu_1,2173,31.591,0.008775,stage_runtime,False,elasticnet_cv,224,10115
100,all_benchmarks_no_unimol_20260423_223505,chemml_organic_density,500,26.795,0.007443,stage_runtime,False,elasticnet_cv,241,10115
101,all_benchmarks_no_unimol_20260423_223505,tdc_half_life_obach,667,6.981,0.001939,stage_runtime,False,elasticnet_cv,512,10115


Selector scaling fit (log10 seconds vs log10 n_molecules): slope=1.312, intercept=-1.435, corr=0.718


Selector-time source diagnostics

,selector_seconds_source,datasets
1,stage_runtime,78
0,first_model_elapsed_proxy,25


`first_model_elapsed_proxy` indicates an estimated upper bound for selector time when explicit stage timings were not saved for that run.

In [76]:
reconstructed_heatmap_source = SUMMARY_METRICS.pivot_table(index="dataset", columns="model", values=RMSE_COL, aggfunc="mean").reset_index()

def _heatmap_value_count(frame):
    if frame is None or frame.empty:
        return 0
    table = frame.copy()
    if "dataset" not in table.columns:
        first_col = table.columns[0]
        table = table.rename(columns={first_col: "dataset"})
    if "dataset" not in table.columns:
        return 0
    value_table = table.set_index("dataset").apply(pd.to_numeric, errors="coerce")
    return int(np.isfinite(value_table.to_numpy(dtype=float)).sum())

heatmap_source = reconstructed_heatmap_source.copy()
heatmap_source_label = "reconstructed_from_summary_metrics"

if TEST_RMSE_PIVOT is not None and not TEST_RMSE_PIVOT.empty:
    pivot_file_source = TEST_RMSE_PIVOT.copy()
    if "dataset" not in pivot_file_source.columns:
        first_col = pivot_file_source.columns[0]
        pivot_file_source = pivot_file_source.rename(columns={first_col: "dataset"})

    file_value_count = _heatmap_value_count(pivot_file_source)
    reconstructed_value_count = _heatmap_value_count(reconstructed_heatmap_source)

    if file_value_count >= reconstructed_value_count:
        heatmap_source = pivot_file_source
        heatmap_source_label = "test_rmse_pivot.csv"
    else:
        display_note(
            "Using reconstructed RMSE heatmap source from merged metrics because it contains more populated "
            f"cells than `test_rmse_pivot.csv` ({reconstructed_value_count} vs {file_value_count})."
        )

if "dataset" not in heatmap_source.columns:
    first_col = heatmap_source.columns[0]
    heatmap_source = heatmap_source.rename(columns={first_col: "dataset"})

heatmap_df = heatmap_source.set_index("dataset")
fig = px.imshow(
    heatmap_df,
    aspect="auto",
    color_continuous_scale="Viridis_r",
    labels={"x": "Model", "y": "Dataset", "color": RMSE_COL},
    text_auto='.3f',
    title=f"Held-out RMSE by dataset and model ({heatmap_source_label})",
)
fig.update_layout(height=max(450, 60 + 40 * len(heatmap_df.index)))
fig.show()

box_df = SUMMARY_METRICS.copy()
fig = px.box(
    box_df,
    x="model",
    y=RMSE_COL,
    color="workflow" if "workflow" in box_df.columns else None,
    points="all",
    title="RMSE distribution across datasets",
)
fig.update_layout(xaxis_title="Model", yaxis_title=RMSE_COL)
fig.show()

delta_df = SUMMARY_METRICS.copy()
error_col = pick_column(delta_df.columns, ["error", "Error"])
if error_col is not None:
    delta_df = delta_df.loc[
        delta_df[error_col].isna() | (delta_df[error_col].astype(str).str.strip() == "")
    ].copy()

delta_df[RMSE_COL] = pd.to_numeric(delta_df[RMSE_COL], errors="coerce")
delta_df = delta_df.loc[delta_df[RMSE_COL].notna()].copy()

if delta_df.empty:
    display_note("No valid RMSE rows available for delta-from-best model boxplot.")
else:
    dataset_best_rmse = delta_df.groupby("dataset", as_index=False)[RMSE_COL].min().rename(columns={RMSE_COL: "dataset_best_rmse"})
    delta_df = delta_df.merge(dataset_best_rmse, on="dataset", how="left")
    delta_df["delta_rmse_from_best"] = delta_df[RMSE_COL] - delta_df["dataset_best_rmse"]

    model_order = (
        delta_df.groupby("model", as_index=False)["delta_rmse_from_best"]
        .median()
        .sort_values("delta_rmse_from_best", ascending=True)["model"]
        .tolist()
    )

    positive_deltas = delta_df.loc[delta_df["delta_rmse_from_best"] > 0, "delta_rmse_from_best"]
    delta_floor = float(positive_deltas.min() * 0.5) if not positive_deltas.empty else 1e-12
    delta_floor = max(delta_floor, 1e-12)
    delta_df["delta_rmse_from_best_log10"] = np.log10(delta_df["delta_rmse_from_best"] + delta_floor)

    fig = px.box(
        delta_df,
        x="model",
        y="delta_rmse_from_best_log10",
        points="all",
        category_orders={"model": model_order},
        title="Model RMSE delta from dataset-best model (log10 scale, lower is better)",
    )
    fig.update_layout(
        xaxis_title="Model",
        yaxis_title=f"log10(RMSE delta + {delta_floor:.2e})",
        xaxis_tickangle=45,
    )
    fig.show()


In [77]:
if PREDICTIONS is not None and not PREDICTIONS.empty:
    pred_df = PREDICTIONS.copy()
    dataset_col = pick_column(pred_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(pred_df.columns, ["model", "Model"])
    observed_col = pick_column(pred_df.columns, ["observed", "y_true", "actual"])
    predicted_col = pick_column(pred_df.columns, ["predicted", "y_pred", "prediction"])
    split_col = pick_column(pred_df.columns, ["split", "Split"])

    if all(col is not None for col in [dataset_col, model_col, observed_col, predicted_col]):
        pred_df = pred_df.rename(columns={
            dataset_col: "dataset",
            model_col: "model",
            observed_col: "observed",
            predicted_col: "predicted",
        })
        if split_col is not None:
            pred_df = pred_df.rename(columns={split_col: "split"})
            pred_df = pred_df.loc[pred_df["split"].astype(str).str.lower() == "test"].copy()

        pred_df["observed"] = pd.to_numeric(pred_df["observed"], errors="coerce")
        pred_df["predicted"] = pd.to_numeric(pred_df["predicted"], errors="coerce")
        pred_df = pred_df.loc[pred_df["observed"].notna() & pred_df["predicted"].notna()].copy()

        metrics_best = SUMMARY_METRICS.copy()
        metrics_error_col = pick_column(metrics_best.columns, ["error", "Error"])
        if metrics_error_col is not None:
            metrics_best = metrics_best.loc[
                metrics_best[metrics_error_col].isna() | (metrics_best[metrics_error_col].astype(str).str.strip() == "")
            ].copy()
        metrics_best[RMSE_COL] = pd.to_numeric(metrics_best[RMSE_COL], errors="coerce")
        metrics_best = metrics_best.loc[metrics_best[RMSE_COL].notna()].copy()

        if metrics_best.empty:
            display_note("No successful metric rows available to resolve best model per dataset for held-out scatter.")
        else:
            best_model_by_dataset = (
                metrics_best.sort_values(RMSE_COL, ascending=True)
                .groupby("dataset", as_index=False)
                .first()[["dataset", "model"]]
                .rename(columns={"model": "best_model"})
            )

            pred_df = pred_df.merge(best_model_by_dataset, on="dataset", how="inner")
            pred_df = pred_df.loc[pred_df["model"].astype(str) == pred_df["best_model"].astype(str)].copy()

            if pred_df.empty:
                display_note("No held-out prediction rows matched each dataset's best model.")
            else:
                if len(pred_df) > int(prediction_sample_cap):
                    dataset_count = max(1, pred_df["dataset"].nunique())
                    per_dataset_cap = max(1, int(prediction_sample_cap // dataset_count))
                    pred_df = (
                        pred_df.groupby("dataset", group_keys=False)
                        .apply(lambda frame: frame.sample(n=min(len(frame), per_dataset_cap), random_state=42))
                        .reset_index(drop=True)
                    )

                fig = px.scatter(
                    pred_df,
                    x="observed",
                    y="predicted",
                    color="best_model",
                    facet_col="dataset",
                    facet_col_wrap=3,
                    opacity=0.70,
                    title="Observed vs predicted (held-out rows) for each dataset's best model",
                    hover_data=["dataset", "best_model"],
                )
                fig.update_traces(marker=dict(size=5))
                fig.update_layout(legend_title_text="Best model")
                fig.show()
    else:
        display_note("Predictions table exists, but the expected observed/predicted columns were not found.")
else:
    display_note("No aggregate predictions table was found for this run.")

if GA_HISTORY is not None and not GA_HISTORY.empty:
    ga_df = GA_HISTORY.copy()
    dataset_col = pick_column(ga_df.columns, ["dataset", "dataset_id", "dataset_name"])
    model_col = pick_column(ga_df.columns, ["model", "Model"])
    generation_col = pick_column(ga_df.columns, ["generation", "Generation"])
    best_col = pick_column(ga_df.columns, ["best_fitness", "best_rmse", "best_score", "Best fitness"])
    if all(col is not None for col in [dataset_col, model_col, generation_col, best_col]):
        ga_df = ga_df.rename(columns={dataset_col: "dataset", model_col: "model", generation_col: "generation", best_col: "best_value"})
        fig = px.line(ga_df, x="generation", y="best_value", color="model", facet_col="dataset", facet_col_wrap=3, title="GA convergence history")
        fig.update_layout(yaxis_title=best_col)
        fig.show()
    else:
        display_note("GA history exists, but the expected columns were not found for plotting.")


No aggregate predictions table was found for this run.

In [78]:
# Ensemble diagnostics: method-aware value-add vs best base model

ENSEMBLE_EPS = 1e-12
METHOD_CFA = "CFA (Combinatorial Fusion)"
METHOD_OOF = "OOF stacking (RidgeCV)"
METHOD_INV_RMSE = "Model average (inverse train RMSE)"
EXPECTED_ENSEMBLE_METHODS = [METHOD_CFA, METHOD_OOF, METHOD_INV_RMSE]
BASELINE_LABEL = "Best base model"


def classify_ensemble_method(model_name, workflow_name=""):
    model_text = str(model_name or "").strip().lower()
    workflow_text = str(workflow_name or "").strip().lower()

    if model_text.startswith("cfa (") or "cfa" in workflow_text:
        return METHOD_CFA

    if model_text.startswith("ensemble (") or "ensemble" in workflow_text:
        if "oof stacking" in model_text:
            return METHOD_OOF
        if "weighted average" in model_text and "inverse" in model_text and "rmse" in model_text:
            return METHOD_INV_RMSE
    return None


perf_df = SUMMARY_METRICS.copy()
error_col = pick_column(perf_df.columns, ["error", "Error"])
if error_col is not None:
    perf_df = perf_df.loc[perf_df[error_col].isna() | (perf_df[error_col].astype(str).str.strip() == "")].copy()

perf_df[RMSE_COL] = pd.to_numeric(perf_df[RMSE_COL], errors="coerce")
perf_df = perf_df.loc[perf_df[RMSE_COL].notna()].copy()

if "workflow" not in perf_df.columns:
    perf_df["workflow"] = ""

perf_df["ensemble_method"] = [
    classify_ensemble_method(model_name, workflow_name)
    for model_name, workflow_name in zip(perf_df["model"], perf_df["workflow"])
]

ensemble_df = perf_df.loc[perf_df["ensemble_method"].notna()].copy()
base_df = perf_df.loc[perf_df["ensemble_method"].isna()].copy()

if perf_df.empty:
    display_note("No valid RMSE rows available for ensemble-vs-base diagnostics.")
elif ensemble_df.empty or base_df.empty:
    display_note("Could not compare ensemble methods vs best base model because one side is missing.")
else:
    method_availability = (
        ensemble_df.groupby("ensemble_method", dropna=False)
        .agg(rows=("dataset", "size"), datasets=("dataset", "nunique"))
        .reset_index()
    )
    method_availability = (
        method_availability.set_index("ensemble_method")
        .reindex(EXPECTED_ENSEMBLE_METHODS)
        .reset_index()
        .rename(columns={"index": "ensemble_method"})
    )
    method_availability[["rows", "datasets"]] = method_availability[["rows", "datasets"]].fillna(0).astype(int)

    display_note("### Ensemble Method Coverage")
    display(method_availability)

    # Optional weight diagnostics when ensemble weight artifacts are present.
    ensemble_best_method_by_dataset = (
        ensemble_df.sort_values(RMSE_COL, ascending=True)
        .groupby("dataset", as_index=False)
        .first()[["dataset", "ensemble_method", "model"]]
        .rename(columns={"ensemble_method": "selected_ensemble_method", "model": "selected_ensemble_model"})
    )
    selected_method_lookup = dict(zip(ensemble_best_method_by_dataset["dataset"], ensemble_best_method_by_dataset["selected_ensemble_method"]))

    ensemble_weight_rows = []
    for weights_path in sorted(RUN_DIR.glob("*/ensemble_weights.csv")):
        dataset_name = weights_path.parent.name
        try:
            wdf = pd.read_csv(weights_path)
        except Exception:
            continue
        if wdf.empty:
            continue
        model_col = pick_column(wdf.columns, ["model", "Model"])
        weight_col = pick_column(wdf.columns, ["weight", "Weight"])
        abs_col = pick_column(wdf.columns, ["abs normalized contribution", "Abs normalized contribution", "abs_normalized_contribution"])
        workflow_col = pick_column(wdf.columns, ["workflow", "Workflow"])
        if model_col is None or weight_col is None:
            continue

        local = pd.DataFrame(
            {
                "dataset": dataset_name,
                "model": wdf[model_col].astype(str),
                "weight": pd.to_numeric(wdf[weight_col], errors="coerce"),
                "abs_normalized_contribution": pd.to_numeric(wdf[abs_col], errors="coerce") if abs_col is not None else np.nan,
                "workflow": wdf[workflow_col].astype(str) if workflow_col is not None else "",
                "selected_ensemble_method": selected_method_lookup.get(dataset_name, np.nan),
            }
        )
        local = local.loc[local["model"].str.strip().ne("") & local["weight"].notna()].copy()
        if not local.empty:
            ensemble_weight_rows.append(local)

    if ensemble_weight_rows:
        ensemble_weights_df = pd.concat(ensemble_weight_rows, ignore_index=True)
        ensemble_weights_df["abs_weight"] = ensemble_weights_df["weight"].abs()

        usage_summary = (
            ensemble_weights_df.groupby(["selected_ensemble_method", "model"], dropna=False)
            .agg(
                datasets=("dataset", "nunique"),
                total_rows=("dataset", "size"),
                mean_weight=("weight", "mean"),
                median_weight=("weight", "median"),
                mean_abs_weight=("abs_weight", "mean"),
                median_abs_weight=("abs_weight", "median"),
                max_abs_weight=("abs_weight", "max"),
                mean_abs_contribution=("abs_normalized_contribution", "mean"),
                median_abs_contribution=("abs_normalized_contribution", "median"),
            )
            .reset_index()
            .sort_values(["selected_ensemble_method", "mean_abs_contribution", "mean_abs_weight", "datasets"], ascending=[True, False, False, False])
        )

        display_note("### Ensemble Weight Usage Statistics (when `ensemble_weights.csv` is present)")
        display(usage_summary)

        weight_plot_df = ensemble_weights_df.copy()
        weight_plot_df["selected_ensemble_method"] = weight_plot_df["selected_ensemble_method"].fillna("Unknown")
        fig = px.box(
            weight_plot_df,
            x="model",
            y="weight",
            color="selected_ensemble_method",
            points="all",
            title="Distribution of ensemble weights by selected ensemble method",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(xaxis_title="Model", yaxis_title="Weight", xaxis_tickangle=45)
        fig.show()
    else:
        display_note("No `ensemble_weights.csv` files were found for this run; skipping weight diagnostics.")

    method_best = (
        ensemble_df.groupby(["dataset", "ensemble_method"], as_index=False)[RMSE_COL]
        .min()
        .rename(columns={RMSE_COL: "ensemble_rmse"})
    )
    base_best = (
        base_df.groupby("dataset", as_index=False)[RMSE_COL]
        .min()
        .rename(columns={RMSE_COL: "best_base_rmse"})
    )

    comparison = method_best.merge(base_best, on="dataset", how="inner")
    comparison["delta_rmse_ensemble_minus_best_base"] = comparison["ensemble_rmse"] - comparison["best_base_rmse"]
    comparison["relative_delta_vs_best_base"] = np.where(
        comparison["best_base_rmse"].abs() > ENSEMBLE_EPS,
        comparison["delta_rmse_ensemble_minus_best_base"] / comparison["best_base_rmse"],
        np.nan,
    )
    comparison["ensemble_beats_best_base"] = comparison["delta_rmse_ensemble_minus_best_base"] < -ENSEMBLE_EPS
    comparison["ensemble_ties_best_base"] = comparison["delta_rmse_ensemble_minus_best_base"].abs() <= ENSEMBLE_EPS

    if comparison.empty:
        display_note("No overlapping datasets with both ensemble-method and base-model RMSE values were found.")
    else:
        summary_rows = []
        for method_name in EXPECTED_ENSEMBLE_METHODS:
            local = comparison.loc[comparison["ensemble_method"] == method_name].copy()
            if local.empty:
                summary_rows.append(
                    {
                        "ensemble_method": method_name,
                        "datasets_compared": 0,
                        "ensemble_beats_base_count": 0,
                        "ensemble_ties_base_count": 0,
                        "ensemble_loses_to_base_count": 0,
                        "ensemble_beats_base_fraction": np.nan,
                        "mean_delta_rmse": np.nan,
                        "median_delta_rmse": np.nan,
                        "std_delta_rmse": np.nan,
                        "min_delta_rmse": np.nan,
                        "max_delta_rmse": np.nan,
                        "mean_relative_delta": np.nan,
                        "median_relative_delta": np.nan,
                    }
                )
                continue

            beats = int((local["delta_rmse_ensemble_minus_best_base"] < -ENSEMBLE_EPS).sum())
            ties = int((local["delta_rmse_ensemble_minus_best_base"].abs() <= ENSEMBLE_EPS).sum())
            losses = int((local["delta_rmse_ensemble_minus_best_base"] > ENSEMBLE_EPS).sum())
            total = int(len(local))

            summary_rows.append(
                {
                    "ensemble_method": method_name,
                    "datasets_compared": total,
                    "ensemble_beats_base_count": beats,
                    "ensemble_ties_base_count": ties,
                    "ensemble_loses_to_base_count": losses,
                    "ensemble_beats_base_fraction": beats / total if total else np.nan,
                    "mean_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].mean()),
                    "median_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].median()),
                    "std_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].std(ddof=1)) if total >= 2 else np.nan,
                    "min_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].min()),
                    "max_delta_rmse": float(local["delta_rmse_ensemble_minus_best_base"].max()),
                    "mean_relative_delta": float(local["relative_delta_vs_best_base"].mean(skipna=True)),
                    "median_relative_delta": float(local["relative_delta_vs_best_base"].median(skipna=True)),
                }
            )

        method_vs_base_summary = pd.DataFrame(summary_rows).sort_values(
            ["ensemble_beats_base_fraction", "mean_delta_rmse"],
            ascending=[False, True],
        ).reset_index(drop=True)

        display_note("### Ensemble Method Delta vs Best Base Model (across datasets)")
        display(method_vs_base_summary)
        display(comparison.sort_values(["ensemble_method", "delta_rmse_ensemble_minus_best_base"], ascending=[True, True]).reset_index(drop=True))

        fig = px.box(
            comparison,
            x="ensemble_method",
            y="delta_rmse_ensemble_minus_best_base",
            color="ensemble_method",
            points="all",
            title="Ensemble RMSE delta vs best base model across datasets (by method)",
        )
        fig.add_hline(y=0.0, line_dash="dash", line_color="black")
        fig.update_layout(
            xaxis_title="Ensemble method",
            yaxis_title="ensemble RMSE - best base RMSE (negative is better)",
            showlegend=False,
        )
        fig.show()

        # Pairwise superiority between ensemble methods on overlapping datasets.
        method_matrix = method_best.pivot(index="dataset", columns="ensemble_method", values="ensemble_rmse")
        pairwise_rows = []
        for i, method_a in enumerate(EXPECTED_ENSEMBLE_METHODS):
            if method_a not in method_matrix.columns:
                continue
            for method_b in EXPECTED_ENSEMBLE_METHODS[i + 1 :]:
                if method_b not in method_matrix.columns:
                    continue
                pair_df = method_matrix[[method_a, method_b]].dropna().copy()
                if pair_df.empty:
                    continue
                delta = pair_df[method_a] - pair_df[method_b]
                wins_a = int((delta < -ENSEMBLE_EPS).sum())
                wins_b = int((delta > ENSEMBLE_EPS).sum())
                ties = int((delta.abs() <= ENSEMBLE_EPS).sum())
                compared = int(len(pair_df))
                pairwise_rows.append(
                    {
                        "method_a": method_a,
                        "method_b": method_b,
                        "datasets_compared": compared,
                        "method_a_wins": wins_a,
                        "method_b_wins": wins_b,
                        "ties": ties,
                        "method_a_win_fraction": wins_a / compared if compared else np.nan,
                        "mean_delta_rmse_a_minus_b": float(delta.mean()),
                        "median_delta_rmse_a_minus_b": float(delta.median()),
                    }
                )

        if pairwise_rows:
            pairwise_df = pd.DataFrame(pairwise_rows).sort_values(
                ["datasets_compared", "method_a_win_fraction", "mean_delta_rmse_a_minus_b"],
                ascending=[False, False, True],
            ).reset_index(drop=True)

            superiority_score = {m: {"wins": 0, "losses": 0, "ties": 0, "comparisons": 0} for m in EXPECTED_ENSEMBLE_METHODS}
            for _, row in pairwise_df.iterrows():
                a = row["method_a"]
                b = row["method_b"]
                aw = int(row["method_a_wins"])
                bw = int(row["method_b_wins"])
                tw = int(row["ties"])
                superiority_score[a]["wins"] += aw
                superiority_score[a]["losses"] += bw
                superiority_score[a]["ties"] += tw
                superiority_score[a]["comparisons"] += int(row["datasets_compared"])
                superiority_score[b]["wins"] += bw
                superiority_score[b]["losses"] += aw
                superiority_score[b]["ties"] += tw
                superiority_score[b]["comparisons"] += int(row["datasets_compared"])

            superiority_table = pd.DataFrame(
                [
                    {
                        "ensemble_method": method_name,
                        "pairwise_wins": payload["wins"],
                        "pairwise_losses": payload["losses"],
                        "pairwise_ties": payload["ties"],
                        "pairwise_compared": payload["comparisons"],
                        "pairwise_win_fraction": payload["wins"] / payload["comparisons"] if payload["comparisons"] else np.nan,
                    }
                    for method_name, payload in superiority_score.items()
                ]
            ).sort_values(["pairwise_win_fraction", "pairwise_wins"], ascending=[False, False]).reset_index(drop=True)

            display_note("### Ensemble Method Pairwise Superiority (head-to-head)")
            display(pairwise_df)
            display(superiority_table)
        else:
            display_note("No overlapping datasets had RMSE values for pairwise method-vs-method ensemble comparisons.")

        # Dataset-level winner summary including the best base model.
        best_base_series = base_best.set_index("dataset")["best_base_rmse"]
        winner_matrix = method_matrix.copy()
        winner_matrix[BASELINE_LABEL] = best_base_series
        winner_matrix = winner_matrix.dropna(how="all")

        if winner_matrix.empty:
            display_note("Could not compute cross-dataset winner summary (no overlap between ensemble methods and base model).")
        else:
            winner_weights = {name: 0.0 for name in [BASELINE_LABEL] + EXPECTED_ENSEMBLE_METHODS}
            considered_datasets = 0
            for _, row in winner_matrix.iterrows():
                row_valid = row.dropna()
                if row_valid.empty:
                    continue
                considered_datasets += 1
                best_val = float(row_valid.min())
                winners = [name for name, value in row_valid.items() if abs(float(value) - best_val) <= ENSEMBLE_EPS]
                if not winners:
                    continue
                share = 1.0 / len(winners)
                for winner in winners:
                    winner_weights[winner] = winner_weights.get(winner, 0.0) + share

            winner_summary = pd.DataFrame(
                [
                    {
                        "method_or_baseline": name,
                        "dataset_best_weighted_wins": float(winner_weights.get(name, 0.0)),
                        "dataset_best_weighted_win_fraction": (
                            float(winner_weights.get(name, 0.0)) / considered_datasets if considered_datasets else np.nan
                        ),
                        "datasets_considered": considered_datasets,
                    }
                    for name in [BASELINE_LABEL] + EXPECTED_ENSEMBLE_METHODS
                ]
            ).sort_values(["dataset_best_weighted_win_fraction", "dataset_best_weighted_wins"], ascending=[False, False]).reset_index(drop=True)

            display_note("### Cross-Dataset Winner Summary (best RMSE among base + ensemble methods)")
            display(winner_summary)


Could not compare ensemble methods vs best base model because one side is missing.